In [2]:
import os
# Keep TF's default BFC allocator (no cuda_malloc_async pool).
# Needs a fresh kernel: the allocator is fixed when TF first touches the GPU.
os.environ.pop("TF_GPU_ALLOCATOR", None)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras

for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)

# GPU memory leak fix. On this TF 2.14 / Python 3.11 stack an EagerTensor never releases a
# tensor stored as an attribute on it, and Keras' MultiHeadAttention stores two on every eager
# call (the softmax's `_keras_logits` and `_keras_mask`). Training steps run in a tf.function
# and are unaffected, but validate() and ensemble sampling run eagerly: each validation pass
# stranded ~580 MiB of attention scores, so the 8 GB card OOM'd after ~9 epochs (~3 h).
# Dropping the attributes changes no numbers -- nothing here reads them.
_MHA = keras.layers.MultiHeadAttention
if not getattr(_MHA, "_eager_leak_patched", False):
    _orig_masked_softmax = _MHA._masked_softmax

    def _masked_softmax_no_leak(self, attention_scores, attention_mask=None):
        out = _orig_masked_softmax(self, attention_scores, attention_mask)
        if tf.executing_eagerly():
            out.__dict__.clear()
        return out

    _MHA._masked_softmax = _masked_softmax_no_leak
    _MHA._eager_leak_patched = True

print("GPU ready:", tf.config.list_physical_devices("GPU"))


GPU ready: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Does the Sky-Video Branch Pay for Itself?

### A calibration-aware ablation of multimodal diffusion for ultra-short-term probabilistic PV forecasting

**Target artefact:** 6-page conference paper (IEEE PES GM / IEEE PVSC / ICMLA / NeurIPS Climate Change AI workshop).

**Dataset:** SKIPP'D (Stanford, 64x64 sky images + 30 kW rooftop PV, 1-min cadence), 15-minute horizon.

**Baseline under test:** _PV-MM-diffusion: An end-to-end multi-modal diffusion model for ultra-short-term
probabilistic photovoltaic forecasting_, Huang, Shao, Ke, Gao & Mazzoni, **Applied Energy** (2026)
— [ScienceDirect S030626192600468X](https://www.sciencedirect.com/science/article/pii/S030626192600468X).

---

## What this notebook is, and what `ddim.ipynb` is

`ddim.ipynb` is a careful **reimplementation** of the baseline: the same Random-Shift PV Multi-Modal
Attention (RS-PV-MMA) block, the same coupled U-Net with a PV 1-D branch and a factorized 2D/1D sky branch,
the same three-mode empty-frame placeholder scheme, the same SKIPP'D 30 kW array, the same 15-min horizon,
the same CRPS / Winkler / VGG-CS metric set, and the same smart-persistence and two-stage baselines.
It swaps the sampler for DDIM. It contains **no research question**, and its `LITERATURE_REFERENCE` list is
literally empty, so it makes no claim against any published number.

**Reimplementing a 2026 Applied Energy paper is not a contribution.** This notebook turns that engineering
asset into a publishable study by asking a question the baseline and its neighbours leave open, and by
answering it with an evaluation protocol that is stricter than the one they use.


## 1. The gap, and the evidence that it is a gap

### 1.1 A live disagreement in the literature

Three papers on the _same dataset and horizon_ disagree about whether generating future sky imagery is
worth its cost:

| Work                                                                                                                        | Position on pixel-space future-sky generation                                                                                                                                                                                                                                                                                           |
| --------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **SkyGPT** (Nie et al., _Adv. Appl. Energy_, 2024) — [arXiv:2306.11682](https://arxiv.org/abs/2306.11682)                   | Generates sky video with a physics-constrained VideoGPT, then maps to PV with a U-Net. CRPS 2.81 kW, WS 26.70. **But its own error analysis names the image→power "nowcast" mapping, not the video prediction, as the dominant error source**, and finds a domain gap between generated and real frames that fine-tuning barely closes. |
| **PV-MM-diffusion** (Huang et al., _Appl. Energy_, 2026)                                                                    | Doubles down: jointly denoises future sky frames _and_ future PV in one model. Reports CRPS 2.63 kW (−18%) and WS 21.46 kW (−40%) vs. SkyGPT→U-Net.                                                                                                                                                                                     |
| **Benchmarking DL for irradiance estimation from sky images** (2025) — [arXiv:2503.21966](https://arxiv.org/pdf/2503.21966) | Finds that **synthetically predicted future frames do not necessarily improve irradiance forecasts over direct prediction**, and that frame errors compound across the horizon.                                                                                                                                                         |

Nobody has settled this, because **nobody has run the controlled ablation**. PV-MM-diffusion ablates the
_training_ diffusion trajectory (500 vs. 1000 steps; linear vs. cosine schedule) and the placeholder modes.
It never removes the sky-generation branch itself. SkyGPT never trains a matched no-video comparator.
The benchmarking paper studies deterministic irradiance, not a probabilistic diffusion forecaster.

> **Gap 1 — unanswered:** _in a multimodal diffusion PV forecaster, how much probabilistic skill comes from
> generating the future sky, versus merely conditioning on the observed past sky — and what does the
> generation branch cost in parameters and sampling latency?_

### 1.2 "Reliable intervals" is asserted, never verified

Every paper in this line scores probabilistic quality with **CRPS, Winkler, and at most one coverage number
at one nominal level**. None of them reports the diagnostics that actually separate _calibration_ from
_sharpness_:

- verification rank / PIT histograms and a reliability index,
- reliability diagrams across nominal levels, with average coverage error,
- the Hersbach CRPS decomposition into **reliability − resolution + uncertainty**,
- spread–skill ratio.

This matters because in the wider ML-weather literature **diffusion ensembles are repeatedly found to be
underdispersed** — overconfident, with spread–skill ratio < 1 and U-shaped rank histograms
(StormCast, [arXiv:2408.10958](https://arxiv.org/pdf/2408.10958); diffusion downscaling; GraphCast-Perturbed
as reported in [GenCast, arXiv:2312.15796](https://arxiv.org/pdf/2312.15796)).
A low CRPS is perfectly compatible with a badly calibrated ensemble, because CRPS rewards sharpness and
calibration jointly. **No published sky-image PV diffusion model has been audited for calibration.**

> **Gap 2 — unanswered:** _are diffusion-based PV ensembles calibrated, or are their headline CRPS/Winkler
> gains bought with overconfidence? And can a cheap post-hoc correction recover calibration without
> giving back the CRPS?_

### 1.3 Two scoring defects inherited by the whole sub-literature

1. **Biased CRPS.** The standard empirical estimator with an $M^2$ spread denominator is biased for finite
   ensemble size $M$ and only converges as $M\to\infty$; a CRPS from a 10-member ensemble is **not
   comparable** to one from a 30- or 50-member ensemble (Ferro 2014, _fair scores for ensemble forecasts_;
   see also [AIFS-CRPS](https://www.nature.com/articles/s44387-026-00073-7) on the almost-fair variant).
   SkyGPT uses 10 scenarios, `ddim.ipynb` uses 30, PV-MM-diffusion uses its own budget — and all use the
   biased estimator, including `compute_crps` in `ddim.ipynb`. Cross-paper CRPS comparisons in this area
   are therefore **not sound as reported**.
2. **No significance testing, and dependent samples.** Sky-image windows are extracted with a small stride
   within a day, so neighbouring test cases are near-duplicates. Scores are reported as bare point values
   with no interval and no paired test, over a handful of test days.

> **Gap 3 — unanswered:** _the sub-literature's ranking of methods has never been shown to survive an
> unbiased score and a resampling test that respects day-level dependence._

### 1.4 The inference-time budget is unexplored

PV-MM-diffusion ablates the _training_ trajectory length. The operationally decisive quantity is different:
**how few denoising steps can you afford at inference before the ensemble stops being calibrated?**
The general time-series literature warns that "aggressive compression risks calibration drift", but for
solar nowcasting — where the forecast must be delivered inside the 2-minute cadence it updates on — nobody
has mapped the latency × CRPS × calibration surface. `ddim.ipynb` sweeps DDIM steps but records
**only latency**, never rescoring accuracy, so its sweep cannot answer this.

> **Gap 4 — unanswered:** _what is the Pareto frontier between sampling latency and probabilistic forecast
> quality for a sky-image PV diffusion model?_

### 1.5 Gaps deliberately **not** pursued here

Named so reviewers see they were considered and rejected for good reason:

- **Cross-site generalization / transfer.** Genuinely important, but already well worked
  ([SPIRIT, arXiv:2502.10307](https://arxiv.org/html/2502.10307v1); physics-informed transfer,
  _Renew. Sustain. Energy Rev._ 2024) — and impossible to study honestly on SKIPP'D, which is one site.
- **Ramp-event forecasting as the headline.** Occupied as of 2026 by _Ultra-Short-Term Photovoltaic Ramp
  Forecasting via Physics-Guided Generative Sky Video Prediction_ (Wang & You,
  [arXiv:2605.03165](https://arxiv.org/abs/2605.03165)). We use ramp windows only as a conditioning
  stratum, not as the contribution.
- **A new architecture.** The field does not need a fifth attention variant; it needs to know whether the
  fourth one was necessary.


## 2. Research questions, hypotheses, and falsification criteria

Stated **before** any number is computed, so the study cannot be retrofitted to its results.

|         | Question                                                                                               | Hypothesis                                                                                                                               | Falsified if                                                                                        |
| ------- | ------------------------------------------------------------------------------------------------------ | ---------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------- |
| **RQ1** | Does co-denoising future sky frames improve probabilistic PV skill over conditioning on past sky only? | **H1**: the gain is small relative to its cost — ≤ 5% fair-CRPS — and not significant under a day-block paired test.                     | A0 beats A2 by > 5% fair CRPS with a day-block bootstrap CI on the paired difference excluding 0.   |
| **RQ2** | Is the diffusion PV ensemble calibrated?                                                               | **H2**: it is **underdispersed** — spread–skill ratio < 1, U-shaped rank histogram, Hersbach reliability a non-trivial share of CRPS.    | Rank histogram flat (reliability index within its resampling null band) and spread–skill ratio ≈ 1. |
| **RQ3** | Can cheap post-hoc recalibration fix it without losing CRPS?                                           | **H3**: validation-fitted spread inflation + split-conformal intervals cut average coverage error by ≥ 50% at equal or better fair CRPS. | Recalibration fails to improve coverage error, or costs > 2% fair CRPS.                             |
| **RQ4** | How few sampling steps survive?                                                                        | **H4**: fair CRPS is flat down to ≈ 10 steps, then degrades; **calibration degrades earlier than CRPS**.                                 | CRPS and calibration degrade at the same step budget, or neither degrades down to 1 step.           |

**A negative result on RQ1 is a publishable result**, and is the honest outcome if H1 holds: it tells the
field that the expensive half of a 2026 Applied Energy architecture is not carrying its weight on this
benchmark. The contribution does not depend on winning.

### Contributions claimed

1. **C1 — The first controlled ablation of the sky-generation branch** in a multimodal diffusion PV
   forecaster: four variants on one backbone, one dataset, one training budget, reported jointly with
   parameters and sampling latency.
2. **C2 — The first calibration audit** of diffusion-based sky-image PV forecasting: rank/PIT histograms,
   reliability diagrams, Hersbach decomposition, spread–skill.
3. **C3 — A corrected evaluation protocol** for this sub-literature: fair (unbiased) CRPS, day-block
   bootstrap CIs, paired significance tests, ensemble-size sensitivity.
4. **C4 — The latency × quality Pareto frontier** at inference, separating the step budget at which CRPS
   degrades from the earlier one at which calibration does.
5. **C5 — A drop-in recalibration recipe** (spread inflation + split conformal) that costs one validation
   pass and no retraining.


## 3. Setup — reuse the model and data code from `ddim.ipynb`

`ddim.ipynb` is the single source of truth for the data pipeline, the RS-PV-MMA block, the coupled U-Net,
the diffusion wrapper, the DDIM sampler and the baselines. We execute its **definition** cells only and
stop before its training drivers, so nothing is duplicated and nothing is retrained by accident.


In [3]:
import json, os, sys, time, math, gc, datetime, itertools
from typing import Dict, List, Tuple, Optional, Any

BASE_NOTEBOOK = os.environ.get("BASE_NOTEBOOK", "ddim.ipynb")
# Everything before this marker is definitions + data loading. The marked cell and everything
# after it reload ddim.ipynb's trained models and run its full test evaluation; executing those
# here leaves that model, sampler and evaluation tensors resident on the GPU for the whole
# ablation run (and overwrites ddim.ipynb's result files), which OOMs A0 training.
STOP_MARKER = "# ---- Reload trained models"


def exec_definition_cells(path: str, stop_marker: str) -> int:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Could not find {path}. Set the BASE_NOTEBOOK environment variable to the "
            "path of ddim.ipynb (this notebook reuses its model and data code)."
        )
    nb = json.load(open(path, encoding="utf-8"))
    code = ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
    stop = next((i for i, src in enumerate(code) if stop_marker in src), None)
    # Never fall through silently: a stale marker would execute the whole base notebook.
    if stop is None:
        raise RuntimeError(
            f"Stop marker {stop_marker!r} not found in {path}. Update STOP_MARKER to the "
            "first cell that reloads/evaluates models."
        )
    for n, src in enumerate(code[:stop]):
        exec(compile(src, f"<{os.path.basename(path)}:cell{n}>", "exec"), globals())
    return stop


_n_exec = exec_definition_cells(BASE_NOTEBOOK, STOP_MARKER)
print(f"\nExecuted {_n_exec} definition cells from {BASE_NOTEBOOK}.")

for _required in [
    "CONFIG_15M",
    "CoupledUNet",
    "CoupledMultimodalDiffusion",
    "DDIMSampler",
    "RandomShiftPVMMA",
    "PV1DBlock",
    "SinusoidalTimeEmbedding",
    "TimestepMLP",
    "create_tf_dataset",
    "get_train_val_indices",
    "extract_day_aware_sequences",
    "normalize_pv",
    "denormalize_pv",
    "normalize_images",
    "trainval_images_mmap",
    "trainval_pv_raw",
    "times_trainval_raw",
    "test_images_mmap",
    "test_pv_raw",
    "times_test_raw",
    "SmartPersistenceBaseline",
    "PV_RATED_CAPACITY_KW",
    "DIFFUSION_STEPS_T",
]:
    if _required not in globals():
        raise RuntimeError(
            f"'{_required}' was not defined by {BASE_NOTEBOOK}; the reuse marker may be stale."
        )
print("All required symbols are available.")

TensorFlow version: 2.14.1
Available GPUs: 1
  Configured memory growth for: /physical_device:GPU:0
Environment initialized successfully.
15-Minute Horizon DDIM Configuration initialized:
  in] hist=8 frames, fut=8 frames (lead time = 15 min), total sequence=16 frames, RS-PV-MMA windows=[1, 4, 8], batch size=8, DDIM steps=50
Found valid SKIPP'D dataset at: /home/luqman-khan-57/content/data/extracted
trainval :  349372 records | images (64, 64, 3) uint8 | PV kW min/mean/max = 0.00/13.28/29.59 | 2017-03-09 06:47:00 -> 2019-10-26 17:59:20
test     :   14003 records | images (64, 64, 3) uint8 | PV kW min/mean/max = 0.00/12.59/29.57 | 2017-09-15 07:11:00 -> 2019-10-19 18:15:30
Sequence extractor ready.
tf.data pipeline builder defined successfully.
Time embedding layers created.
RS-PV-MMA attention layer defined successfully.
Coupled U-Net backbone defined successfully.
Coupled multimodal diffusion model defined successfully.
DDIM fast inference sampler (graph-compiled) defined successfully

### 3.1 Run configuration

`QUICK_MODE = True` runs the whole study end-to-end on a small subset so every cell can be smoke-tested in
minutes. **Every number in the paper must come from a `QUICK_MODE = False` run.** The flag is recorded in
the results file so no quick-mode number can be mistaken for a final one.


In [4]:
QUICK_MODE = False  # <-- set False for the numbers that go in the paper


class Run:
    seed_list = [0] if QUICK_MODE else [0, 1, 2]  # training seeds per variant
    epochs = 2 if QUICK_MODE else 60
    patience = 2 if QUICK_MODE else 10
    base_dim = 8 if QUICK_MODE else 16
    batch_size = 4
    train_stride = 12 if QUICK_MODE else 3  # window stride on trainval
    test_stride = 30 if QUICK_MODE else 6  # window stride on test
    max_train_windows = 400 if QUICK_MODE else None
    max_test_windows = 64 if QUICK_MODE else None
    M_eval = 8 if QUICK_MODE else 50  # ensemble members for the headline table
    M_sweep = [2, 4, 8] if QUICK_MODE else [2, 5, 10, 20, 30, 50]
    steps_eval = 10 if QUICK_MODE else 20  # DDIM steps for the headline table
    steps_sweep = [2, 5, 10] if QUICK_MODE else [1, 2, 5, 10, 20, 50, 100]
    n_boot = 200 if QUICK_MODE else 5000
    sample_batch = 4
    ramp_threshold = (
        0.20  # |dP| / rated capacity over the horizon defines a ramp window
    )
    nominal_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    results_path = "results_gap_quick.json" if QUICK_MODE else "results_gap.json"
    figure_dir = "figures_gap"


os.makedirs(Run.figure_dir, exist_ok=True)
CONFIG_15M.batch_size = Run.batch_size

print(f"QUICK_MODE = {QUICK_MODE}")
print(f"  seeds={Run.seed_list}  epochs={Run.epochs}  base_dim={Run.base_dim}")
print(f"  M_eval={Run.M_eval}  steps_eval={Run.steps_eval}  bootstrap={Run.n_boot}")
print(f"  results -> {Run.results_path}")
if QUICK_MODE:
    print("\n  *** QUICK MODE: these numbers are smoke tests, not results. ***")

QUICK_MODE = False
  seeds=[0, 1, 2]  epochs=60  base_dim=16
  M_eval=50  steps_eval=20  bootstrap=5000
  results -> results_gap.json


---

## 4. Contribution C3 — a scoring suite this sub-literature does not currently use

`ddim.ipynb` (and, as far as their published tables show, every paper it reimplements) scores probabilistic
quality with a **biased** CRPS, one Winkler score and one coverage number. That cannot distinguish a sharp
but overconfident forecast from a well-calibrated one, and it cannot support a comparison between models
evaluated at different ensemble sizes.

This section implements, with self-tests against closed-form values:

- **`crps_ensemble`** — both the biased ($M^2$) and the **fair / unbiased** ($M(M-1)$) estimator
  (Ferro 2014). Reported side by side so the size of the bias is visible.
- **`hersbach_decomposition`** — CRPS = reliability − resolution + uncertainty (Hersbach 2000), which
  separates _how miscalibrated_ the ensemble is from _how much information_ it has.
- **`rank_histogram`** — verification rank (PIT) histogram with randomized tie-breaking, plus a reliability
  index and a resampling null band for it.
- **`spread_skill_ratio`** — with the $\sqrt{(M+1)/M}$ finite-ensemble correction, so 1.0 means calibrated.
- **`reliability_curve`** — empirical coverage across nine nominal levels, and average coverage error.
- **`day_block_bootstrap` / `paired_day_block_test`** — resampling at the level of **calendar days**,
  because overlapping intra-day windows are not independent samples.


In [5]:
import numpy as np

# ----------------------------------------------------------------------------------
# CRPS: biased (M^2) and fair / unbiased (M(M-1)) empirical estimators
# ----------------------------------------------------------------------------------


def _crps_terms(y_true: np.ndarray, ens: np.ndarray):
    """y_true (N,T); ens (N,T,M). Returns (skill_term, spread_numerator, M)."""
    y = np.asarray(y_true, dtype=np.float64)
    x = np.asarray(ens, dtype=np.float64)
    if y.ndim == 1:
        y, x = y[:, None], x[:, None, :]
    M = x.shape[-1]
    skill = np.mean(np.abs(x - y[..., None]), axis=-1)  # (N,T)
    xs = np.sort(x, axis=-1)
    w = 2.0 * np.arange(1, M + 1) - M - 1.0
    spread_num = np.sum(xs * w, axis=-1)  # (N,T), = 0.5*sum_ij|xi-xj|*2/1
    return skill, spread_num, M


def crps_per_case(y_true, ens, fair: bool = True) -> np.ndarray:
    """Per-(case, lead-time) CRPS in the units of y. Returns (N, T)."""
    skill, spread_num, M = _crps_terms(y_true, ens)
    if M < 2:
        return skill
    denom = (M * (M - 1)) if fair else (M * M)
    return skill - spread_num / denom


def crps_ensemble(y_true, ens, fair: bool = True) -> float:
    return float(np.nanmean(crps_per_case(y_true, ens, fair=fair)))


def crps_gaussian(y, mu, sigma) -> float:
    """Closed-form CRPS of N(mu, sigma^2) at y -- used only to self-test the estimators."""
    from scipy.stats import norm

    z = (np.asarray(y, dtype=float) - mu) / sigma
    return float(
        np.mean(
            sigma * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1.0 / np.sqrt(np.pi))
        )
    )


# ----------------------------------------------------------------------------------
# Hersbach (2000) CRPS decomposition:  CRPS = Reliability - Resolution + Uncertainty
# ----------------------------------------------------------------------------------


def hersbach_decomposition(y_true, ens) -> Dict[str, float]:
    """
    y_true (N,) or (N,T); ens matching with a trailing member axis. All cases are pooled.
    Returns reliability, resolution, uncertainty, crps_potential and the reconstructed crps.
    """
    y = np.asarray(y_true, dtype=np.float64).reshape(-1)
    x = np.asarray(ens, dtype=np.float64).reshape(len(y), -1)
    good = np.isfinite(y) & np.isfinite(x).all(axis=1)
    y, x = y[good], x[good]
    N, M = x.shape
    if N == 0 or M < 2:
        return {
            k: float("nan")
            for k in (
                "reliability",
                "resolution",
                "uncertainty",
                "crps_potential",
                "crps_reconstructed",
            )
        }
    xs = np.sort(x, axis=1)

    alpha = np.zeros((N, M + 1))  # width of the sub-interval BELOW the observation
    beta = np.zeros((N, M + 1))  # width of the sub-interval ABOVE the observation

    lo, hi = xs[:, :-1], xs[:, 1:]  # interior intervals i = 1..M-1
    width = hi - lo
    above = y[:, None] > hi
    below = y[:, None] < lo
    inside = ~above & ~below
    alpha[:, 1:M] = np.where(above, width, np.where(inside, y[:, None] - lo, 0.0))
    beta[:, 1:M] = np.where(below, width, np.where(inside, hi - y[:, None], 0.0))

    # Outlier bins: i = 0 (obs below the lowest member) and i = M (obs above the highest)
    out_low = y < xs[:, 0]
    out_high = y > xs[:, -1]
    beta[:, 0] = np.where(out_low, xs[:, 0] - y, 0.0)
    alpha[:, M] = np.where(out_high, y - xs[:, -1], 0.0)

    a_bar, b_bar = alpha.mean(axis=0), beta.mean(axis=0)
    p = np.arange(M + 1, dtype=np.float64) / M
    crps = float(np.sum(a_bar * p**2 + b_bar * (1.0 - p) ** 2))

    g = a_bar + b_bar
    o = np.divide(b_bar, g, out=np.full_like(g, np.nan), where=g > 0)
    # Hersbach's convention for the two outlier bins, whose widths are unbounded
    f_low, f_high = float(out_low.mean()), float(out_high.mean())
    o[0] = f_low
    g[0] = b_bar[0] / f_low if f_low > 0 else 0.0
    o[M] = 1.0 - f_high
    g[M] = a_bar[M] / f_high if f_high > 0 else 0.0
    valid = np.isfinite(o) & np.isfinite(g)

    reliability = float(np.sum(g[valid] * (o[valid] - p[valid]) ** 2))
    crps_pot = float(np.sum(g[valid] * o[valid] * (1.0 - o[valid])))

    ysort = np.sort(y)
    w = 2.0 * np.arange(1, len(y) + 1) - len(y) - 1.0
    uncertainty = (
        float(np.sum(ysort * w) / (len(y) * (len(y) - 1)))
        if len(y) > 1
        else float("nan")
    )

    return {
        "reliability": reliability,
        "resolution": uncertainty - crps_pot,
        "uncertainty": uncertainty,
        "crps_potential": crps_pot,
        "crps_reconstructed": reliability + crps_pot,
    }


# ----------------------------------------------------------------------------------
# Rank (PIT) histogram, reliability index, spread-skill ratio
# ----------------------------------------------------------------------------------


def rank_histogram(y_true, ens, seed: int = 0) -> Dict[str, Any]:
    """Verification rank in 1..M+1 with randomized tie-breaking; flat == calibrated."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y_true, dtype=np.float64).reshape(-1)
    x = np.asarray(ens, dtype=np.float64).reshape(len(y), -1)
    good = np.isfinite(y) & np.isfinite(x).all(axis=1)
    y, x = y[good], x[good]
    N, M = x.shape
    below = np.sum(x < y[:, None], axis=1)
    ties = np.sum(x == y[:, None], axis=1)
    ranks = 1 + below + np.array([rng.integers(0, t + 1) for t in ties])
    counts = np.bincount(ranks, minlength=M + 2)[1 : M + 2]
    freq = counts / max(counts.sum(), 1)
    uniform = 1.0 / (M + 1)
    ri = float(np.sum(np.abs(freq - uniform)))

    # Null band for the reliability index under perfect calibration, same N and M
    null = np.empty(400)
    for b in range(400):
        c = rng.multinomial(N, np.full(M + 1, uniform))
        null[b] = np.sum(np.abs(c / N - uniform))
    return {
        "counts": counts.tolist(),
        "frequencies": freq.tolist(),
        "reliability_index": ri,
        "ri_null_p95": float(np.percentile(null, 95)),
        "calibrated": bool(ri <= np.percentile(null, 95)),
        "outlier_fraction": float(freq[0] + freq[-1]),
        "expected_outlier_fraction": float(2.0 / (M + 1)),
        "n_cases": int(N),
        "n_members": int(M),
    }


def spread_skill_ratio(y_true, ens) -> Dict[str, float]:
    """Corrected so that 1.0 means calibrated; < 1 means underdispersed (overconfident)."""
    y = np.asarray(y_true, dtype=np.float64).reshape(-1)
    x = np.asarray(ens, dtype=np.float64).reshape(len(y), -1)
    M = x.shape[1]
    spread = float(np.sqrt(np.nanmean(np.var(x, axis=1, ddof=1))))
    skill = float(np.sqrt(np.nanmean((np.nanmean(x, axis=1) - y) ** 2)))
    corr = np.sqrt((M + 1.0) / M)
    return {
        "spread": spread,
        "rmse_of_mean": skill,
        "ratio_corrected": float(spread * corr / skill) if skill > 0 else float("nan"),
        "underdispersed": bool(skill > 0 and spread * corr / skill < 1.0),
    }


# ----------------------------------------------------------------------------------
# Interval scores: coverage across nominal levels, Winkler, sharpness
# ----------------------------------------------------------------------------------


def interval_stats(y_true, ens, level: float) -> Dict[str, Any]:
    """Central prediction interval at nominal coverage `level` (e.g. 0.9)."""
    y = np.asarray(y_true, dtype=np.float64)
    x = np.asarray(ens, dtype=np.float64)
    if y.ndim == 1:
        y, x = y[:, None], x[:, None, :]
    a = 1.0 - level
    L = np.percentile(x, 100 * a / 2, axis=-1)
    U = np.percentile(x, 100 * (1 - a / 2), axis=-1)
    covered = (y >= L) & (y <= U)
    width = U - L
    winkler = (
        width
        + np.where(y < L, 2 * (L - y) / a, 0.0)
        + np.where(y > U, 2 * (y - U) / a, 0.0)
    )
    return {
        "level": float(level),
        "coverage": float(np.nanmean(covered)),
        "coverage_error": float(np.nanmean(covered) - level),
        "winkler": float(np.nanmean(winkler)),
        "width": float(np.nanmean(width)),
        "winkler_per_case": np.nanmean(winkler, axis=1),
        "lower": L,
        "upper": U,
    }


def reliability_curve(y_true, ens, levels=None) -> Dict[str, Any]:
    levels = levels or [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    rows = [interval_stats(y_true, ens, lv) for lv in levels]
    cov = [r["coverage"] for r in rows]
    return {
        "levels": list(levels),
        "coverage": cov,
        "width": [r["width"] for r in rows],
        "winkler": [r["winkler"] for r in rows],
        "ace": float(
            np.mean(np.abs(np.array(cov) - np.array(levels)))
        ),  # average coverage error
        "signed_ace": float(np.mean(np.array(cov) - np.array(levels))),
    }


# ----------------------------------------------------------------------------------
# Resampling that respects day-level dependence between overlapping windows
# ----------------------------------------------------------------------------------


def day_block_bootstrap(
    per_case: np.ndarray,
    day_ids: np.ndarray,
    n_boot: int = 5000,
    alpha: float = 0.05,
    seed: int = 0,
):
    """Bootstrap the mean of a per-case score by resampling whole calendar days."""
    rng = np.random.default_rng(seed)
    v = np.asarray(per_case, dtype=np.float64).reshape(len(day_ids), -1).mean(axis=1)
    days = np.unique(day_ids)
    by_day = [np.where(day_ids == d)[0] for d in days]
    boots = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, len(days), size=len(days))
        boots[b] = np.nanmean(np.concatenate([v[by_day[i]] for i in pick]))
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {
        "mean": float(np.nanmean(v)),
        "ci_low": float(lo),
        "ci_high": float(hi),
        "n_days": int(len(days)),
        "n_cases": int(len(v)),
    }


def paired_day_block_test(
    score_a: np.ndarray,
    score_b: np.ndarray,
    day_ids: np.ndarray,
    n_boot: int = 5000,
    seed: int = 0,
):
    """Paired test of mean(A) - mean(B) with days as the resampling unit. Lower score = better."""
    rng = np.random.default_rng(seed)
    a = np.asarray(score_a, dtype=np.float64).reshape(len(day_ids), -1).mean(axis=1)
    b = np.asarray(score_b, dtype=np.float64).reshape(len(day_ids), -1).mean(axis=1)
    d = a - b
    days = np.unique(day_ids)
    by_day = [np.where(day_ids == dd)[0] for dd in days]
    boots = np.empty(n_boot)
    for i in range(n_boot):
        pick = rng.integers(0, len(days), size=len(days))
        boots[i] = np.nanmean(np.concatenate([d[by_day[j]] for j in pick]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = 2.0 * min(float(np.mean(boots >= 0)), float(np.mean(boots <= 0)))
    return {
        "delta": float(np.nanmean(d)),
        "delta_pct": (
            float(100.0 * np.nanmean(d) / np.nanmean(b))
            if np.nanmean(b) != 0
            else float("nan")
        ),
        "ci_low": float(lo),
        "ci_high": float(hi),
        "p_value": float(min(p, 1.0)),
        "significant": bool(lo > 0 or hi < 0),
    }


print("Scoring suite defined.")

Scoring suite defined.


### 4.1 Self-tests — the metrics must be right before any of them is reported

Each assertion below has a closed-form or analytically known answer. If this cell passes, the scoring code
is trustworthy; if it fails, no result downstream means anything.


In [6]:
def _self_test_scoring():
    rng = np.random.default_rng(0)

    # 1. Perfect ensemble (all members equal the observation) -> CRPS = 0
    y = rng.random((40, 8)) * 10
    assert (
        abs(crps_ensemble(y, np.repeat(y[..., None], 16, axis=-1), fair=False)) < 1e-9
    )
    print("[OK] 1  CRPS of a perfect ensemble is 0.")

    # 2. Degenerate (M=1-like) ensemble -> CRPS reduces to MAE
    assert (
        abs(
            crps_ensemble(y, np.repeat((y + 1.0)[..., None], 12, axis=-1), fair=False)
            - 1.0
        )
        < 1e-9
    )
    print("[OK] 2  CRPS of a constant-offset ensemble equals MAE.")

    # 3. Fair CRPS converges to the analytic Gaussian CRPS and is (near) unbiased in M;
    #    the biased estimator is inflated at small M.
    obs = rng.standard_normal(6000)
    analytic = crps_gaussian(obs, mu=0.0, sigma=1.0)
    line = []
    for M in (5, 10, 50, 200):
        ens = rng.standard_normal((6000, 1, M))
        f = crps_ensemble(obs[:, None], ens, fair=True)
        b = crps_ensemble(obs[:, None], ens, fair=False)
        line.append((M, f, b))
        assert (
            abs(f - analytic) < 0.02
        ), f"fair CRPS off at M={M}: {f:.4f} vs {analytic:.4f}"
    assert (
        line[0][2] > line[-1][2] + 0.01
    ), "biased CRPS should shrink towards fair as M grows"
    print(
        f"[OK] 3  fair CRPS ~ analytic {analytic:.4f} at every M; biased inflates at small M:"
    )
    for M, f, b in line:
        print(
            f"         M={M:>3d}  fair={f:.4f}  biased={b:.4f}  bias=+{100*(b-f)/f:5.2f}%"
        )

    # 4. Hersbach reconstruction matches the direct CRPS, and a calibrated ensemble has
    #    near-zero reliability while a shrunken (overconfident) one does not.
    ens_cal = obs[:, None] * 0 + rng.standard_normal((6000, 40))
    dec = hersbach_decomposition(obs, ens_cal)
    assert (
        abs(
            dec["crps_reconstructed"]
            - crps_ensemble(obs[:, None], ens_cal[:, None, :], fair=False)
        )
        < 0.01
    )
    ens_over = ens_cal * 0.35
    dec_over = hersbach_decomposition(obs, ens_over)
    assert dec_over["reliability"] > 5 * max(dec["reliability"], 1e-6)
    print(
        f"[OK] 4  Hersbach reconstructs CRPS; reliability {dec['reliability']:.4f} (calibrated) "
        f"vs {dec_over['reliability']:.4f} (overconfident)."
    )

    # 5. Rank histogram flags a calibrated ensemble as flat and a shrunken one as U-shaped
    rh_cal = rank_histogram(obs, ens_cal)
    rh_over = rank_histogram(obs, ens_over)
    assert rh_cal["calibrated"] and not rh_over["calibrated"]
    assert rh_over["outlier_fraction"] > 3 * rh_cal["outlier_fraction"]
    print(
        f"[OK] 5  rank histogram: calibrated RI={rh_cal['reliability_index']:.4f} "
        f"(null p95 {rh_cal['ri_null_p95']:.4f}); overconfident RI={rh_over['reliability_index']:.4f}, "
        f"outliers {rh_over['outlier_fraction']:.3f} vs expected {rh_over['expected_outlier_fraction']:.3f}."
    )

    # 6. Spread-skill ratio is ~1 when calibrated and < 1 when overconfident
    ss_cal, ss_over = spread_skill_ratio(obs, ens_cal), spread_skill_ratio(
        obs, ens_over
    )
    assert 0.93 < ss_cal["ratio_corrected"] < 1.07 and ss_over["ratio_corrected"] < 0.6
    print(
        f"[OK] 6  spread-skill: {ss_cal['ratio_corrected']:.3f} (calibrated) "
        f"vs {ss_over['ratio_corrected']:.3f} (overconfident)."
    )

    # 7. Empirical coverage tracks the nominal level for a calibrated ensemble
    rc = reliability_curve(obs[:, None], ens_cal[:, None, :])
    assert rc["ace"] < 0.03, rc
    print(f"[OK] 7  reliability curve ACE = {rc['ace']:.4f} for a calibrated ensemble.")

    # 8. Paired day-block test finds a real difference and not a fake one
    dids = np.repeat(np.arange(60), 20)
    base = rng.random(1200)
    assert paired_day_block_test(base + 0.25, base, dids, n_boot=400)["significant"]
    assert not paired_day_block_test(
        base + rng.normal(0, 0.01, 1200), base, dids, n_boot=400
    )["significant"]
    print("[OK] 8  paired day-block test has power and does not fire on noise.")

    print("\nAll scoring self-tests passed.")


_self_test_scoring()

[OK] 1  CRPS of a perfect ensemble is 0.
[OK] 2  CRPS of a constant-offset ensemble equals MAE.


[OK] 3  fair CRPS ~ analytic 0.5618 at every M; biased inflates at small M:
         M=  5  fair=0.5601  biased=0.6729  bias=+20.15%
         M= 10  fair=0.5621  biased=0.6185  bias=+10.05%
         M= 50  fair=0.5621  biased=0.5734  bias=+ 2.01%
         M=200  fair=0.5621  biased=0.5649  bias=+ 0.50%
[OK] 4  Hersbach reconstructs CRPS; reliability 0.0006 (calibrated) vs 0.1120 (overconfident).
[OK] 5  rank histogram: calibrated RI=0.0743 (null p95 0.0785); overconfident RI=0.8970, outliers 0.458 vs expected 0.049.
[OK] 6  spread-skill: 1.004 (calibrated) vs 0.356 (overconfident).
[OK] 7  reliability curve ACE = 0.0259 for a calibrated ensemble.
[OK] 8  paired day-block test has power and does not fire on noise.

All scoring self-tests passed.


---

## 5. Contribution C1 — the ablation ladder

Four variants, one backbone family, one dataset, one training budget. They differ **only** in how much
sky information reaches the PV denoiser and how expensively it gets there.

|        | Variant                                 | Future sky frames                 | Image loss | Sky encoder run               | What it isolates                                        |
| ------ | --------------------------------------- | --------------------------------- | ---------- | ----------------------------- | ------------------------------------------------------- |
| **A0** | `joint-gen` (baseline reimplementation) | **co-denoised** with PV           | yes        | every denoising step          | the full PV-MM-diffusion proposal                       |
| **A1** | `persist-frame`                         | last observed frame, **repeated** | no         | every denoising step          | value of _generating_ future sky, at identical capacity |
| **A2** | `cond-pv`                               | none                              | no         | **once**, cached across steps | value of _conditioning_ on past sky, cheaply            |
| **A3** | `pv-only`                               | none                              | no         | not run                       | lower bound: no imagery at all                          |

The ladder is nested: A3 ⊂ A2 ⊂ A1 ⊂ A0 in sky information and in compute. Reading it left to right
attributes skill to each increment.

**Why A1 is the decisive comparison.** A1 is byte-for-byte the same coupled U-Net as A0, with the same
RS-PV-MMA blocks and -- verified at run time below -- an **identical parameter count**. The only change is
that its future image slots hold a persistence placeholder instead of a denoised sample, and its loss has
no image term. If A0 does not beat A1, the sky **generation** branch is not earning its cost, whatever the
sky **conditioning** is worth.

**A2 is sized to be parameter-comparable to A0, not smaller.** `PVConditionalDiffusion` is built at twice
the width of the joint backbone precisely so that A0 vs. A2 cannot be dismissed as "the bigger model won".
What separates them is then not _how much_ compute there is but **where it is spent**: A0 runs a full
spatio-temporal U-Net over 64x64x3 frames at _every_ denoising step, while A2 runs its sky encoder _once_
and then denoises a 1-D sequence. A3 is the only deliberately small variant and serves as the no-imagery
floor.

**Deliberate design choices, stated so reviewers can check them.**

- All four variants train on **mode 0 only** (both modalities always present). The baseline's three-mode
  placeholder scheme is a separate contribution about missing modalities; mixing it in would confound the
  ablation. A0 is therefore retrained here rather than reused from `ddim.ipynb`.
- A2's encoder pools sky features spatially before attention. That is not a simplification we invented —
  `RandomShiftPVMMA` in the baseline does `reduce_mean(img_feats, axis=[2,3])` before attending, so A2
  discards exactly the spatial detail the baseline's own fusion already discards.
- Variants are matched on **epochs, data, optimizer and early-stopping rule**. A0/A1 are additionally
  matched exactly on parameter count (asserted below), and A2 approximately; A3 is the small floor.
  Parameter counts and sampling latency are **reported as costs in every results table**, never hidden.


In [7]:
# ----------------------------------------------------------------------------------
# Sky-history encoder (A2): runs ONCE per forecast, not once per denoising step
# ----------------------------------------------------------------------------------


class SkyHistoryEncoder(layers.Layer):
    """Encodes the observed history frames into (B, T_hist, ctx_dim) context tokens."""

    def __init__(self, base_dim: int = 16, ctx_dim: int = 128, **kwargs):
        super().__init__(**kwargs)
        d = base_dim
        self.c1 = layers.Conv2D(d, 3, strides=2, padding="same")  # 64 -> 32
        self.c2 = layers.Conv2D(d * 2, 3, strides=2, padding="same")  # 32 -> 16
        self.c3 = layers.Conv2D(d * 4, 3, strides=2, padding="same")  # 16 -> 8
        self.c4 = layers.Conv2D(d * 8, 3, strides=2, padding="same")  # 8  -> 4
        self.pool = layers.GlobalAveragePooling2D()
        self.temporal = layers.Conv1D(ctx_dim, 3, padding="same")
        self.proj = layers.Dense(ctx_dim)
        self.norm = layers.LayerNormalization(epsilon=1e-5)

    def call(self, img_hist):
        B, T = tf.shape(img_hist)[0], tf.shape(img_hist)[1]
        x = tf.reshape(img_hist, [B * T, 64, 64, 3])
        for conv in (self.c1, self.c2, self.c3, self.c4):
            x = tf.nn.silu(conv(x))
        x = tf.reshape(self.pool(x), [B, T, -1])
        x = tf.nn.silu(self.temporal(x))
        return self.norm(self.proj(x))


class CrossAttendToContext(layers.Layer):
    """Pre-norm cross-attention from PV tokens to cached sky-history tokens."""

    def __init__(self, embed_dim: int, num_heads: int = 4, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=max(embed_dim // num_heads, 1)
        )
        self.norm_q = layers.LayerNormalization(epsilon=1e-5)
        self.norm_kv = layers.LayerNormalization(epsilon=1e-5)
        self.norm_out = layers.LayerNormalization(epsilon=1e-5)
        self.ctx_proj = layers.Dense(embed_dim)

    def call(self, x, ctx):
        c = self.norm_kv(self.ctx_proj(ctx))
        return self.norm_out(x + self.mha(query=self.norm_q(x), value=c, key=c))


class PVDenoiser(keras.Model):
    """Compact 1-D denoiser over the PV sequence, optionally cross-attending to sky context."""

    def __init__(self, cfg, base_dim: int = 32, use_sky: bool = True, **kwargs):
        super().__init__(**kwargs)
        self.cfg = cfg
        self.use_sky = use_sky
        self.time_embed = SinusoidalTimeEmbedding(dim=base_dim * 2)
        self.time_mlp = TimestepMLP(out_dim=base_dim * 4)
        self.inp = layers.Dense(base_dim)
        self.enc1 = PV1DBlock(base_dim)
        self.enc2 = PV1DBlock(base_dim * 2)
        self.enc3 = PV1DBlock(base_dim * 4)
        self.dec2 = PV1DBlock(base_dim * 2)
        self.dec1 = PV1DBlock(base_dim)
        self.out = layers.Conv1D(1, kernel_size=1)
        if use_sky:
            self.xattn2 = CrossAttendToContext(base_dim * 2)
            self.xattn3 = CrossAttendToContext(base_dim * 4)

    def call(self, inputs, training=None):
        pv_seq, t = inputs["pv_seq"], inputs["timesteps"]
        t_emb = self.time_mlp(self.time_embed(t))
        h1 = self.enc1(self.inp(pv_seq), t_emb)
        h2 = self.enc2(h1, t_emb)
        if self.use_sky:
            h2 = self.xattn2(h2, inputs["sky_ctx"])
        h3 = self.enc3(h2, t_emb)
        if self.use_sky:
            h3 = self.xattn3(h3, inputs["sky_ctx"])
        d2 = self.dec2(tf.concat([h3, h2], axis=-1), t_emb)
        d1 = self.dec1(tf.concat([d2, h1], axis=-1), t_emb)
        return {"eps_pv": self.out(d1)[:, self.cfg.hist_frames :, :]}


print("A2/A3 building blocks defined.")

A2/A3 building blocks defined.


In [8]:
# ----------------------------------------------------------------------------------
# Diffusion wrappers. All variants share the EXACT beta schedule of the baseline.
# ----------------------------------------------------------------------------------


def _make_schedule(timesteps=DIFFUSION_STEPS_T, beta_start=1e-4, beta_end=0.02):
    betas = np.linspace(beta_start, beta_end, timesteps, dtype=np.float32)
    ac = np.cumprod(1.0 - betas, axis=0)
    return betas, ac


class PVConditionalDiffusion(keras.Model):
    """A2 (use_sky=True) and A3 (use_sky=False): diffusion over future PV only."""

    def __init__(
        self,
        cfg,
        base_dim: int = 32,
        use_sky: bool = True,
        ctx_dim: int = 128,
        timesteps: int = DIFFUSION_STEPS_T,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.cfg = cfg
        self.use_sky = use_sky
        self.timesteps = timesteps
        self.denoiser = PVDenoiser(cfg, base_dim=base_dim, use_sky=use_sky)
        self.encoder = (
            SkyHistoryEncoder(base_dim=base_dim, ctx_dim=ctx_dim) if use_sky else None
        )
        _, ac = _make_schedule(timesteps)
        self.alphas_cumprod = tf.constant(ac, dtype=tf.float32)
        self.sqrt_ac = tf.constant(np.sqrt(ac), dtype=tf.float32)
        self.sqrt_1mac = tf.constant(np.sqrt(1.0 - ac), dtype=tf.float32)
        self.loss_tracker = keras.metrics.Mean(name="loss")

    @property
    def metrics(self):
        return [self.loss_tracker]

    def q_sample(self, x0, t, noise):
        a = tf.reshape(tf.gather(self.sqrt_ac, t), [-1, 1, 1])
        b = tf.reshape(tf.gather(self.sqrt_1mac, t), [-1, 1, 1])
        return a * x0 + b * noise

    def encode_sky(self, img_hist):
        return self.encoder(img_hist) if self.use_sky else None

    def _denoise(self, pv_seq, t, sky_ctx):
        inp = {"pv_seq": pv_seq, "timesteps": t}
        if self.use_sky:
            inp["sky_ctx"] = sky_ctx
        return self.denoiser(inp)["eps_pv"]

    def _loss(self, inputs, targets, training):
        pv_fut = targets["pv_fut"]
        B = tf.shape(pv_fut)[0]
        t = tf.random.uniform([B], 0, self.timesteps, dtype=tf.int32)
        eps = tf.random.normal(tf.shape(pv_fut))
        pv_seq = tf.concat([inputs["pv_hist"], self.q_sample(pv_fut, t, eps)], axis=1)
        ctx = self.encode_sky(inputs["img_hist"]) if self.use_sky else None
        return tf.reduce_mean(tf.square(self._denoise(pv_seq, t, ctx) - eps))

    def train_step(self, data):
        inputs, targets = data
        with tf.GradientTape() as tape:
            loss = self._loss(inputs, targets, True)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(
            [(g, v) for g, v in zip(grads, self.trainable_variables) if g is not None]
        )
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def test_step(self, data):
        inputs, targets = data
        self.loss_tracker.update_state(self._loss(inputs, targets, False))
        return {"loss": self.loss_tracker.result()}


class PersistFrameDiffusion(CoupledMultimodalDiffusion):
    """A1: the baseline backbone, but future sky slots hold the last observed frame and
    the image term is removed from the loss. Same parameter count as A0."""

    def _persist(self, img_hist):
        return tf.tile(img_hist[:, -1:], [1, self.cfg.fut_frames, 1, 1, 1])

    def _forward(self, inputs, targets, training):
        pv_fut = targets["pv_fut"]
        B = tf.shape(pv_fut)[0]
        t = tf.random.uniform([B], 0, self.timesteps, dtype=tf.int32)
        eps = tf.random.normal(tf.shape(pv_fut))
        img_seq = tf.concat(
            [inputs["img_hist"], self._persist(inputs["img_hist"])], axis=1
        )
        pv_seq = tf.concat([inputs["pv_hist"], self.q_sample(pv_fut, t, eps)], axis=1)
        pred = self.unet(
            {"img_seq": img_seq, "pv_seq": pv_seq, "timesteps": t}, training=training
        )
        return tf.reduce_mean(tf.square(pred["eps_pv"] - eps))

    def train_step(self, data):
        inputs, targets = data
        with tf.GradientTape() as tape:
            loss = self._forward(inputs, targets, True)
        grads = tape.gradient(loss, self.unet.trainable_variables)
        self.optimizer.apply_gradients(
            [
                (g, v)
                for g, v in zip(grads, self.unet.trainable_variables)
                if g is not None
            ]
        )
        self.loss_tracker.update_state(loss)
        self.pv_loss_tracker.update_state(loss)
        self.img_loss_tracker.update_state(0.0)
        return {
            "loss": self.loss_tracker.result(),
            "pv_loss": self.pv_loss_tracker.result(),
            "img_loss": self.img_loss_tracker.result(),
        }

    def test_step(self, data):
        inputs, targets = data
        loss = self._forward(inputs, targets, False)
        self.loss_tracker.update_state(loss)
        self.pv_loss_tracker.update_state(loss)
        self.img_loss_tracker.update_state(0.0)
        return {
            "loss": self.loss_tracker.result(),
            "pv_loss": self.pv_loss_tracker.result(),
            "img_loss": self.img_loss_tracker.result(),
        }


print("Diffusion wrappers for A0-A3 defined.")

Diffusion wrappers for A0-A3 defined.


In [9]:
# ----------------------------------------------------------------------------------
# One sampling interface for all four variants, so latency is measured comparably
# ----------------------------------------------------------------------------------


def _ddim_timesteps(T: int, num_steps: int) -> np.ndarray:
    return np.linspace(T - 1, 0, num_steps + 1).round().astype(np.int32)


def sample_joint(model, img_hist, pv_hist, num_steps, persist_frames: bool):
    """A0 (persist_frames=False) and A1 (persist_frames=True). Returns normalized PV (B,Tf,1)."""
    B = tf.shape(pv_hist)[0]
    Tf = model.cfg.fut_frames
    ac = model.alphas_cumprod.numpy()
    x_pv = tf.random.normal([B, Tf, 1])
    if persist_frames:
        fixed_img = tf.tile(img_hist[:, -1:], [1, Tf, 1, 1, 1])
    else:
        x_img = tf.random.normal([B, Tf, 64, 64, 3])
    ts = _ddim_timesteps(model.timesteps, num_steps)

    for i in range(len(ts) - 1):
        tc, tn = int(ts[i]), int(ts[i + 1])
        t_vec = tf.fill([B], tc)
        img_fut = fixed_img if persist_frames else x_img
        pred = model.unet(
            {
                "img_seq": tf.concat([img_hist, img_fut], axis=1),
                "pv_seq": tf.concat([pv_hist, x_pv], axis=1),
                "timesteps": t_vec,
            },
            training=False,
        )
        a_c = float(ac[tc])
        sa, s1a = math.sqrt(a_c), math.sqrt(1.0 - a_c)
        x0_pv = (x_pv - s1a * pred["eps_pv"]) / sa
        if not persist_frames:
            x0_img = (x_img - s1a * pred["eps_img"]) / sa
        if tn == 0:
            x_pv = x0_pv
            if not persist_frames:
                x_img = x0_img
        else:
            a_n = float(ac[tn])
            x_pv = math.sqrt(a_n) * x0_pv + math.sqrt(1.0 - a_n) * pred["eps_pv"]
            if not persist_frames:
                x_img = math.sqrt(a_n) * x0_img + math.sqrt(1.0 - a_n) * pred["eps_img"]
    return x_pv


def sample_conditional(model, img_hist, pv_hist, num_steps):
    """A2 / A3. The sky encoder runs ONCE and is reused across every denoising step."""
    B = tf.shape(pv_hist)[0]
    Tf = model.cfg.fut_frames
    ac = model.alphas_cumprod.numpy()
    ctx = model.encode_sky(img_hist) if model.use_sky else None
    x_pv = tf.random.normal([B, Tf, 1])
    ts = _ddim_timesteps(model.timesteps, num_steps)
    for i in range(len(ts) - 1):
        tc, tn = int(ts[i]), int(ts[i + 1])
        eps = model._denoise(tf.concat([pv_hist, x_pv], axis=1), tf.fill([B], tc), ctx)
        a_c = float(ac[tc])
        x0 = (x_pv - math.sqrt(1.0 - a_c) * eps) / math.sqrt(a_c)
        if tn == 0:
            x_pv = x0
        else:
            a_n = float(ac[tn])
            x_pv = math.sqrt(a_n) * x0 + math.sqrt(1.0 - a_n) * eps
    return x_pv


VARIANT_SPEC = {
    "A0_joint_gen": dict(
        family="joint",
        persist=False,
        use_sky=True,
        label="A0 joint-gen (baseline reimpl.)",
    ),
    "A1_persist_frame": dict(
        family="joint",
        persist=True,
        use_sky=True,
        label="A1 persist-frame (no sky generation)",
    ),
    "A2_cond_pv": dict(
        family="cond",
        persist=None,
        use_sky=True,
        label="A2 cond-pv (cached sky encoder)",
    ),
    "A3_pv_only": dict(
        family="cond", persist=None, use_sky=False, label="A3 pv-only (no imagery)"
    ),
}


def build_variant(name: str, cfg, base_dim: int):
    spec = VARIANT_SPEC[name]
    if spec["family"] == "joint":
        unet = CoupledUNet(cfg, base_dim=base_dim)
        cls = PersistFrameDiffusion if spec["persist"] else CoupledMultimodalDiffusion
        return cls(unet, cfg)
    return PVConditionalDiffusion(cfg, base_dim=base_dim * 2, use_sky=spec["use_sky"])


def sample_pv_normalized(name, model, img_hist, pv_hist, num_steps):
    spec = VARIANT_SPEC[name]
    if spec["family"] == "joint":
        return sample_joint(
            model, img_hist, pv_hist, num_steps, persist_frames=bool(spec["persist"])
        )
    return sample_conditional(model, img_hist, pv_hist, num_steps)


def count_parameters(model) -> int:
    return int(sum(np.prod(v.shape) for v in model.trainable_variables))


print(f"Sampling interface ready for variants: {list(VARIANT_SPEC)}")

Sampling interface ready for variants: ['A0_joint_gen', 'A1_persist_frame', 'A2_cond_pv', 'A3_pv_only']


### 5.1 Data splits

Windows are rebuilt here rather than taken from `ddim.ipynb` so the stride, the window cap and the
day-level split are all under this notebook's control and are recorded in the results file. The split is
**by calendar day**: windows inside a day overlap heavily, so a random or sequential window split would
leak near-duplicates from train into validation.


In [10]:
def build_splits(cfg, val_frac: float = 0.1):
    h, f, dates, _, _ = extract_day_aware_sequences(
        trainval_pv_raw,
        times_trainval_raw,
        cfg.hist_frames,
        cfg.fut_frames,
        sample_stride=Run.train_stride,
    )
    days = np.unique(dates)
    rng = np.random.default_rng(RANDOM_SEED)
    n_val = max(1, int(round(val_frac * len(days))))
    val_days = set(days[rng.permutation(len(days))[:n_val]].tolist())
    is_val = np.array([d in val_days for d in dates])
    tr, va = (h[~is_val], f[~is_val]), (h[is_val], f[is_val])
    if Run.max_train_windows:
        k = min(Run.max_train_windows, len(tr[0]))
        sel = rng.permutation(len(tr[0]))[:k]
        tr = (tr[0][sel], tr[1][sel])
        kv = max(1, k // 8)
        selv = rng.permutation(len(va[0]))[: min(kv, len(va[0]))]
        va = (va[0][selv], va[1][selv])
    print(
        f"Train days {len(days) - n_val} / val days {n_val} | "
        f"{len(tr[0])} train windows, {len(va[0])} val windows (stride {Run.train_stride})"
    )
    return tr, va


def build_test_set(cfg):
    h, f, dates, t_fut, t_hist = extract_day_aware_sequences(
        test_pv_raw,
        times_test_raw,
        cfg.hist_frames,
        cfg.fut_frames,
        sample_stride=Run.test_stride,
    )
    if Run.max_test_windows and len(h) > Run.max_test_windows:
        sel = np.linspace(0, len(h) - 1, Run.max_test_windows).round().astype(int)
        h, f, dates, t_fut, t_hist = h[sel], f[sel], dates[sel], t_fut[sel], t_hist[sel]
    day_ids = np.array([str(d) for d in dates])
    print(
        f"Test: {len(h)} windows over {len(np.unique(day_ids))} days (stride {Run.test_stride})"
    )
    return dict(
        h=h, f=f, dates=dates, day_ids=day_ids, times_fut=t_fut, times_hist=t_hist
    )


train_idx, val_idx = build_splits(CONFIG_15M)
TEST = build_test_set(CONFIG_15M)

# Materialise the test tensors once; every variant is scored on exactly these windows.
TEST["pv_true_kw"] = test_pv_raw[TEST["f"]]  # (N, Tf) kW
TEST["pv_hist_kw"] = test_pv_raw[TEST["h"]]  # (N, Th) kW
TEST["img_hist"] = normalize_images(test_images_mmap[TEST["h"]]).astype(np.float32)
TEST["pv_hist"] = normalize_pv(TEST["pv_hist_kw"])[..., None].astype(np.float32)

# Ramp stratum: |P(t+H) - P(now)| exceeding a fraction of rated capacity
delta = np.abs(TEST["pv_true_kw"][:, -1] - TEST["pv_hist_kw"][:, -1])
TEST["is_ramp"] = delta / PV_RATED_CAPACITY_KW > Run.ramp_threshold
print(
    f"Ramp windows: {int(TEST['is_ramp'].sum())}/{len(TEST['is_ramp'])} "
    f"({100*TEST['is_ramp'].mean():.1f}%) at threshold {Run.ramp_threshold:.0%} of rated capacity"
)

Train days 447 / val days 50 | 99473 train windows, 11096 val windows (stride 3)
Test: 2229 windows over 20 days (stride 6)
Ramp windows: 197/2229 (8.8%) at threshold 20% of rated capacity


### 5.2 Training

Identical loop, optimizer, epoch budget and early-stopping rule for every variant and every seed. Wall-clock
training time is recorded, because it is one of the costs the ablation is weighing.


In [11]:
# ==================================================================================
# 5.2 Training -- resumable, checkpointed every epoch, early-stopped on PV loss
# ==================================================================================
# Per (variant, seed) run directory:  ckpt_gap/<name>_seed<seed>_<config-fingerprint>/
#   last_model.npz / last_optim.npz  -> written every epoch (resume point)
#   best_model.npz                   -> written only when validation PV loss improves
#   partial.npz                      -> every MID_EPOCH_CKPT_SECONDS inside an epoch (mid-epoch
#                                       resume point; deleted once that epoch is committed)
#   state.json                       -> epoch, best, patience counter, time, history (commit marker)
# Re-running this cell resumes unfinished runs (from the step they stopped at) and simply
# reloads finished ones.
# The fingerprint includes QUICK_MODE and the training config, so quick-mode checkpoints are
# never reused for the full run.

import hashlib

CKPT_DIR = "ckpt_gap"
VAL_SEED = 1234
RESUME = True  # set False to force retraining from scratch
MID_EPOCH_CKPT_SECONDS = 600  # a crash or restart loses at most ~10 min of an epoch


def _run_fingerprint(cfg) -> str:
    key = dict(
        quick=QUICK_MODE,
        base_dim=Run.base_dim,
        epochs=Run.epochs,
        patience=Run.patience,
        batch=Run.batch_size,
        stride=Run.train_stride,
        max_train=Run.max_train_windows,
        lr=float(cfg.learning_rate),
    )
    return hashlib.sha1(json.dumps(key, sort_keys=True).encode()).hexdigest()[:10]


def _atomic_savez(path, arrays):
    tmp = path[:-4] + ".tmp.npz"
    np.savez(tmp, *arrays)
    os.replace(tmp, path)  # atomic rename: a crash never leaves a half-written checkpoint


def _atomic_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2)
    os.replace(tmp, path)


def _load_npz(path):
    with np.load(path) as z:
        return [z[f"arr_{i}"] for i in range(len(z.files))]


def _assign(variables, arrays, what):
    variables = list(variables)
    if len(variables) != len(arrays):
        raise ValueError(f"{what}: {len(arrays)} arrays for {len(variables)} variables")
    for v, a in zip(variables, arrays):
        if tuple(v.shape) != a.shape:
            raise ValueError(f"{what}: shape {a.shape} does not match {tuple(v.shape)}")
        v.assign(a)


def _opt_vars(opt):
    v = opt.variables
    return list(v() if callable(v) else v)


def _metric_vars(model):
    return [v for m in model.metrics for v in m.variables]


def _save_partial(path, epoch, step, seconds, model, opt):
    """Weights, optimizer, running train metrics and position in ONE atomically renamed file,
    so the pieces of a mid-epoch resume point can never be out of sync with each other."""
    arrays = {"meta": np.array([epoch, step, seconds], dtype=np.float64)}
    for prefix, vs in (("w_", model.weights), ("o_", _opt_vars(opt)), ("m_", _metric_vars(model))):
        arrays.update({f"{prefix}{i}": v.numpy() for i, v in enumerate(vs)})
    tmp = path[:-4] + ".tmp.npz"
    np.savez(tmp, **arrays)
    os.replace(tmp, path)


def _load_partial(path, model, opt):
    with np.load(path) as z:
        part = {
            p: [z[f"{p}{i}"] for i in range(sum(k.startswith(p) for k in z.files))]
            for p in ("w_", "o_", "m_")
        }
        epoch, step, seconds = z["meta"]
    _assign(model.weights, part["w_"], "partial model")
    _assign(_opt_vars(opt), part["o_"], "partial optimizer")
    _assign(_metric_vars(model), part["m_"], "partial metrics")
    return int(epoch), int(step), float(seconds)


def _partial_epoch(path):
    with np.load(path) as z:
        return int(z["meta"][0])


def fit_variant(name: str, seed: int, cfg, verbose_every: int = 20, resume: bool = True):
    run_dir = os.path.join(CKPT_DIR, f"{name}_seed{seed}_{_run_fingerprint(cfg)}")
    os.makedirs(run_dir, exist_ok=True)
    p_last_w = os.path.join(run_dir, "last_model.npz")
    p_last_o = os.path.join(run_dir, "last_optim.npz")
    p_best = os.path.join(run_dir, "best_model.npz")
    p_state = os.path.join(run_dir, "state.json")
    p_part = os.path.join(run_dir, "partial.npz")

    tf.keras.backend.clear_session()
    tf.random.set_seed(seed)
    np.random.seed(seed)

    ds_tr = create_tf_dataset(
        trainval_images_mmap,
        trainval_pv_raw,
        train_idx[0],
        train_idx[1],
        cfg,
        is_training=True,
        mode_weights=(1.0, 0.0, 0.0),
    )
    ds_va = create_tf_dataset(
        trainval_images_mmap,
        trainval_pv_raw,
        val_idx[0],
        val_idx[1],
        cfg,
        is_training=False,
        mode_weights=(1.0, 0.0, 0.0),
    )

    model = build_variant(name, cfg, Run.base_dim)
    opt = keras.optimizers.Adam(learning_rate=cfg.learning_rate)
    model.compile(optimizer=opt)

    # Build all model variables, then the optimizer slots, so both can be restored
    for xb, yb in ds_tr.take(1):
        model.test_step((xb, yb))
    opt.build(model.trainable_variables)
    n_params = count_parameters(model)

    def _meta(state):
        return dict(
            name=name,
            seed=seed,
            params=n_params,
            best_val_loss=state["best"],
            best_epoch=state["best_epoch"],
            train_seconds=state["train_seconds"],
            epochs_run=state["epoch"],
            selection_metric="val_pv_loss",
            run_dir=run_dir,
        )

    state = dict(
        epoch=0,
        best=float("inf"),
        best_epoch=-1,
        bad=0,
        train_seconds=0.0,
        done=False,
        history=[],
    )
    if resume and os.path.exists(p_state):
        with open(p_state, encoding="utf-8") as fh:
            state = json.load(fh)
        if state["done"]:
            _assign(model.weights, _load_npz(p_best), "best model")
            print(
                f"  [{name} s{seed}] already finished; loaded best weights "
                f"(epoch {state['best_epoch']}, val pv {state['best']:.4f})"
            )
            return model, _meta(state)
        _assign(model.weights, _load_npz(p_last_w), "last model")
        _assign(_opt_vars(opt), _load_npz(p_last_o), "optimizer")
        print(
            f"  [{name} s{seed}] resuming after epoch {state['epoch']} "
            f"(best {state['best']:.4f} @ {state['best_epoch']}, patience used {state['bad']})"
        )

    # A partial checkpoint is only valid for the epoch that was in progress; anything else is a
    # leftover from an epoch that was committed afterwards (or from a run we are not resuming).
    start_step = 0
    if os.path.exists(p_part):
        if resume and _partial_epoch(p_part) == state["epoch"] + 1:
            _, start_step, state["train_seconds"] = _load_partial(p_part, model, opt)
            print(f"  [{name} s{seed}] resuming inside epoch {state['epoch'] + 1} at step {start_step}")
        else:
            os.remove(p_part)

    # No data-dependent Python control flow inside; source not introspectable from exec'd cells
    _no_convert = tf.autograph.experimental.do_not_convert
    train_fn = tf.function(_no_convert(model.train_step), reduce_retracing=True)

    def validate():
        # Reseed and run eagerly so every epoch is scored on the SAME (t, eps) draws;
        # changes in val loss then reflect the weights, not diffusion sampling noise.
        tf.random.set_seed(VAL_SEED)
        for m in model.metrics:
            m.reset_state()
        vout = None
        for batch in ds_va:
            vout = model.test_step(batch)
        if vout is None:
            return float("nan")
        # Select every variant on PV loss (A0's total loss also contains the image term)
        return float(vout["pv_loss"] if "pv_loss" in vout else vout["loss"])

    t_mark = time.time()
    for epoch in range(state["epoch"] + 1, Run.epochs + 1):
        # The generator shuffles with the global NumPy RNG when the epoch starts. Seeding it per
        # (seed, epoch) makes a resumed epoch replay the same window order (up to the 256-window
        # shuffle buffer), so skipping the steps already trained lands on the untrained windows.
        np.random.seed(seed * 1000 + epoch)
        skip, start_step = start_step, 0
        if not skip:  # a resumed epoch keeps its restored running-mean train loss
            for m in model.metrics:
                m.reset_state()
        step, t_part = skip, time.time()
        for batch in ds_tr.skip(skip) if skip else ds_tr:
            out = train_fn(batch)
            step += 1
            if time.time() - t_part >= MID_EPOCH_CKPT_SECONDS:
                t_part = time.time()
                _save_partial(
                    p_part, epoch, step, state["train_seconds"] + t_part - t_mark, model, opt
                )
            if verbose_every and step % verbose_every == 0:
                sys.stdout.write(
                    f"\r  [{name} s{seed}] epoch {epoch:02d} step {step:4d} "
                    f"loss {float(out['loss']):.4f}"
                )
                sys.stdout.flush()
        train_loss = float(model.loss_tracker.result())
        val_loss = validate()

        if val_loss < state["best"] - 1e-4:
            state.update(best=val_loss, best_epoch=epoch, bad=0)
            _atomic_savez(p_best, [v.numpy() for v in model.weights])
        else:
            state["bad"] += 1

        now = time.time()
        state["train_seconds"] += now - t_mark
        t_mark = now
        state["epoch"] = epoch
        state["history"].append(
            dict(epoch=epoch, train_loss=train_loss, val_pv_loss=val_loss)
        )
        stop = state["bad"] >= Run.patience
        state["done"] = bool(stop or epoch == Run.epochs)

        # Weights first, state.json last: state.json is the commit marker for this epoch
        _atomic_savez(p_last_w, [v.numpy() for v in model.weights])
        _atomic_savez(p_last_o, [v.numpy() for v in _opt_vars(opt)])
        _atomic_json(p_state, state)
        if os.path.exists(p_part):
            os.remove(p_part)

        gpu_mib = tf.config.experimental.get_memory_info("GPU:0")["current"] / 2**20
        sys.stdout.write(
            f"\r  [{name} s{seed}] epoch {epoch:02d}/{Run.epochs}  "
            f"train {train_loss:.4f}  val(pv) {val_loss:.4f}  GPU in use {gpu_mib:.0f} MiB"
            f"{'  *best*' if state['best_epoch'] == epoch else ''}\n"
        )
        if stop:
            print(
                f"  early stop at epoch {epoch} "
                f"(best {state['best']:.4f} @ {state['best_epoch']})"
            )
            break

    if os.path.exists(p_best):
        _assign(model.weights, _load_npz(p_best), "best model")
    del train_fn, ds_tr, ds_va
    gc.collect()
    return model, _meta(state)


# ----------------------------------------------------------------------------------
# Train (or resume / reload) every variant x seed
# ----------------------------------------------------------------------------------
TRAINED = {}
TRAIN_META = []

for _name in VARIANT_SPEC:
    for _seed in Run.seed_list:
        print(f"\n=== training {VARIANT_SPEC[_name]['label']} | seed {_seed} ===")
        tf.config.experimental.reset_memory_stats("GPU:0")
        try:
            _m, _meta = fit_variant(_name, _seed, CONFIG_15M, resume=RESUME)
        except tf.errors.ResourceExhaustedError as _e:
            raise RuntimeError(
                f"GPU OOM in {_name} seed {_seed} at batch_size={Run.batch_size}. "
                "Restart kernel, re-run (training resumes from the last checkpoint)."
            ) from _e
        TRAINED[(_name, _seed)] = _m
        TRAIN_META.append(_meta)
        peak = tf.config.experimental.get_memory_info("GPU:0")["peak"] / 2**20
        print(f"  params={_meta['params']:,}  best_val_pv={_meta['best_val_loss']:.4f}  "
              f"time={_meta['train_seconds']:.0f}s  GPU peak={peak:.0f} MiB")
        gc.collect()

print()
print(pd.DataFrame(TRAIN_META).drop(columns=["run_dir"]).to_string(index=False))

# The A0-vs-A1 comparison only isolates *generation* if the two share a parameter count. Check it.
_p = {m["name"]: m["params"] for m in TRAIN_META}
assert _p["A0_joint_gen"] == _p["A1_persist_frame"], (
    f"A0 ({_p['A0_joint_gen']:,}) and A1 ({_p['A1_persist_frame']:,}) must be capacity-matched "
    "for the A0-vs-A1 claim to hold."
)
print()
print(
    f"Capacity check: A0 and A1 both have {_p['A0_joint_gen']:,} parameters, so A0 vs A1 "
    "isolates future-sky generation alone."
)
print(
    f"A2 has {_p['A2_cond_pv']:,} ({100*_p['A2_cond_pv']/_p['A0_joint_gen']:.0f}% of A0); "
    f"A3 has {_p['A3_pv_only']:,} ({100*_p['A3_pv_only']/_p['A0_joint_gen']:.0f}% of A0)."
)


=== training A0 joint-gen (baseline reimpl.) | seed 0 ===
  [A0_joint_gen s0] already finished; loaded best weights (epoch 45, val pv 0.0159)
  params=925,412  best_val_pv=0.0159  time=100599s  GPU peak=498 MiB

=== training A0 joint-gen (baseline reimpl.) | seed 1 ===
  [A0_joint_gen s1] already finished; loaded best weights (epoch 45, val pv 0.0150)
  params=925,412  best_val_pv=0.0150  time=89874s  GPU peak=378 MiB

=== training A0 joint-gen (baseline reimpl.) | seed 2 ===
  [A0_joint_gen s2] already finished; loaded best weights (epoch 42, val pv 0.0154)
  params=925,412  best_val_pv=0.0154  time=83905s  GPU peak=378 MiB

=== training A1 persist-frame (no sky generation) | seed 0 ===
  [A1_persist_frame s0] already finished; loaded best weights (epoch 40, val pv 0.0171)
  params=925,412  best_val_pv=0.0171  time=59761s  GPU peak=370 MiB

=== training A1 persist-frame (no sky generation) | seed 1 ===
  [A1_persist_frame s1] already finished; loaded best weights (epoch 41, val pv 0.

---

## 6. Ensemble sampling and the headline comparison (RQ1)


In [12]:
# ----------------------------------------------------------------------------------
# GPU ensemble sampler
# ----------------------------------------------------------------------------------
# The reference samplers in 5.x (sample_joint / sample_conditional) run eagerly, one batch of
# 4 windows and one member at a time: ~557k Python-dispatched U-Net calls per (variant, seed)
# with the GPU ~1% busy (weeks for this section). Here every (window, member) pair is one row
# of a large batch, one DDIM step is a compiled tf.function, and the batch is sized from the
# free VRAM nvidia-smi reports. The DDIM update is the reference's deterministic (eta = 0) rule
# (checked against it to < 0.01 kW with identical noise, for every variant).
# Each member's initial noise is a function of (seed, window, member) only, so the ensemble
# does not depend on how many rows happen to fit on the GPU.
# Finished ensembles are cached in ENS_CACHE_DIR: an interrupted run resumes where it stopped.
import subprocess
import hashlib

ENS_CACHE_DIR = "ens_cache"
VRAM_USE_FRACTION = 0.70  # share of the free VRAM (at first use) the sampler may fill
MAX_SAMPLE_ROWS = 16384  # upper bound on (window, member) rows per compiled call
# TF32 (Ampere default, also used in training) rounds convolutions differently at different
# batch sizes; A0's generated-sky branch amplifies that to ~0.01 kW mean (0.3 kW max) per
# member. False makes ensembles bit-for-bit independent of the VRAM-derived batch size, slower.
SAMPLER_TF32 = True
tf.config.experimental.enable_tensor_float_32_execution(SAMPLER_TF32)
os.makedirs(ENS_CACHE_DIR, exist_ok=True)


def gpu_memory_mib(index: int = 0) -> Tuple[int, int]:
    """(free, total) MiB of one GPU, as nvidia-smi reports them."""
    out = subprocess.run(
        [
            "nvidia-smi",
            f"--id={index}",
            "--query-gpu=memory.free,memory.total",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
    free, total = (int(v) for v in out.strip().split(","))
    return free, total


# Measured once, before the sampler allocates anything: later readings would count the
# memory TensorFlow's allocator already holds (and reuses) as "used".
_free, _total = gpu_memory_mib()
SAMPLER_VRAM_BUDGET_MIB = int(VRAM_USE_FRACTION * _free)
print(
    f"nvidia-smi: {_free} MiB free of {_total} MiB -> sampler budget "
    f"{SAMPLER_VRAM_BUDGET_MIB} MiB ({VRAM_USE_FRACTION:.0%} of free)"
)


def _ddim_coeffs(model, num_steps):
    """Per-step (t, [sqrt a_t, sqrt(1-a_t), sqrt a_next, sqrt(1-a_next)]). The last step
    returns x0 itself, written as a_next = 1 so one compiled function covers every step."""
    ac = model.alphas_cumprod.numpy().astype(np.float64)
    ts = _ddim_timesteps(model.timesteps, num_steps)
    out = []
    for i in range(len(ts) - 1):
        tc, tn = int(ts[i]), int(ts[i + 1])
        a_n = 1.0 if tn == 0 else ac[tn]
        c = [math.sqrt(ac[tc]), math.sqrt(1.0 - ac[tc]), math.sqrt(a_n), math.sqrt(1.0 - a_n)]
        out.append((tf.constant(tc, tf.int32), tf.constant(c, tf.float32)))
    return out


def _member_noise(seed, win, mem, M, shape, stream):
    """N(0, 1) noise for rows (window win[i], member mem[i]); `win` is non-decreasing.
    Drawn per window from a stateless generator keyed on (seed, window, stream)."""
    parts = []
    for w in np.unique(win):
        z = tf.random.stateless_normal([M] + list(shape), seed=[seed, 2 * int(w) + stream])
        parts.append(tf.gather(z, mem[win == w]))
    return tf.concat(parts, axis=0)


class _CompiledSampler:
    """Compiled DDIM sampler for one model. sample() takes each window once plus a row -> window
    index, and expands on the GPU only what the model needs: the sky frames for A0/A1, the
    encoded sky context for A2 (encoded once per window), nothing image-sized for A3."""

    def __init__(self, name, model):
        spec = VARIANT_SPEC[name]
        self.model = model
        self.joint = spec["family"] == "joint"
        self.persist = bool(spec["persist"])
        self.gen_img = self.joint and not self.persist  # A0 also denoises future sky frames
        self.uses_images = self.joint or bool(model.use_sky)
        self.Tf = model.cfg.fut_frames
        # Same as the training step: no data-dependent Python control flow inside, and the
        # source of exec'd cells is not introspectable, so AutoGraph conversion is skipped.
        _no_convert = tf.autograph.experimental.do_not_convert
        if self.joint:
            self._step = tf.function(_no_convert(self._joint_step), reduce_retracing=True)
        else:
            self._encode = (
                tf.function(_no_convert(model.encode_sky), reduce_retracing=True)
                if model.use_sky
                else None
            )
            self._step = tf.function(_no_convert(self._cond_step), reduce_retracing=True)

    def _joint_step(self, x_pv, x_img, img_hist, pv_hist, t, c):
        pred = self.model.unet(
            {
                "img_seq": tf.concat([img_hist, x_img], axis=1),
                "pv_seq": tf.concat([pv_hist, x_pv], axis=1),
                "timesteps": tf.fill([tf.shape(x_pv)[0]], t),
            },
            training=False,
        )
        x_pv = c[2] * ((x_pv - c[1] * pred["eps_pv"]) / c[0]) + c[3] * pred["eps_pv"]
        if self.gen_img:
            x_img = c[2] * ((x_img - c[1] * pred["eps_img"]) / c[0]) + c[3] * pred["eps_img"]
        return x_pv, x_img

    def _cond_step(self, x_pv, pv_hist, ctx, t, c):
        eps = self.model._denoise(
            tf.concat([pv_hist, x_pv], axis=1), tf.fill([tf.shape(x_pv)[0]], t), ctx
        )
        return c[2] * ((x_pv - c[1] * eps) / c[0]) + c[3] * eps

    def upload(self, img_hist, pv_hist, win):
        """The windows covering row -> window map `win`, uploaded once, and local indices."""
        w0, w1 = int(win[0]), int(win[-1]) + 1
        img = tf.constant(img_hist[w0:w1]) if self.uses_images else None
        return img, tf.constant(pv_hist[w0:w1]), tf.constant(win - w0)

    def noise(self, seed, win, mem, M):
        """Initial (x_pv, x_img or None) for the rows (win, mem)."""
        x_pv = _member_noise(seed, win, mem, M, [self.Tf, 1], stream=0)
        x_img = None
        if self.gen_img:
            x_img = _member_noise(seed, win, mem, M, [self.Tf, 64, 64, 3], stream=1)
        return x_pv, x_img

    def sample(self, img_win, pv_win, local, x_pv, x_img, coeffs):
        """img_win (W,Th,64,64,3) or None, pv_win (W,Th,1), local (B,) row -> window index,
        x_pv (B,Tf,1) / x_img (B,Tf,64,64,3) initial noise. Returns normalized PV (B,Tf,1)."""
        pv_hist = tf.gather(pv_win, local)
        if self.joint:
            img_hist = tf.gather(img_win, local)
            if not self.gen_img:  # A1: future sky slots hold the last observed frame
                x_img = tf.tile(img_hist[:, -1:], [1, self.Tf, 1, 1, 1])
            for t, c in coeffs:
                x_pv, x_img = self._step(x_pv, x_img, img_hist, pv_hist, t, c)
            return x_pv
        ctx = None
        if self._encode is not None:  # sky encoder runs once per window, not per member/step
            ctx = tf.gather(self._encode(img_win), local)
        for t, c in coeffs:
            x_pv = self._step(x_pv, pv_hist, ctx, t, c)
        return x_pv

    def run_rows(self, img_hist, pv_hist, flat, M, seed, coeffs):
        """Sample the rows `flat` (indices into the window-major (N*M) grid, non-decreasing)."""
        win, mem = flat // M, flat % M
        x_pv, x_img = self.noise(seed, win, mem, M)
        return self.sample(*self.upload(img_hist, pv_hist, win), x_pv, x_img, coeffs)


_SAMPLERS: Dict[int, _CompiledSampler] = {}
_ROWS: Dict[str, int] = {}  # rows per call, per variant (architecture), once measured


def _sampler_for(name, model) -> _CompiledSampler:
    s = _SAMPLERS.get(id(model))
    if s is None or s.model is not model:
        s = _SAMPLERS[id(model)] = _CompiledSampler(name, model)
    return s


def _rows_for(name, sampler, img_hist, pv_hist, M, coeffs) -> int:
    """Rows per call that fit the VRAM budget: run one probe batch, read the allocator's
    peak, and scale linearly (the fixed overhead counted per row keeps this conservative)."""
    if name in _ROWS:
        return _ROWS[name]
    n_rows = len(pv_hist) * M

    def _run(rows):  # one DDIM step on `rows` rows, inputs created inside the measured region
        flat = np.minimum(np.arange(rows), n_rows - 1)
        sampler.run_rows(img_hist, pv_hist, flat, M, 0, coeffs[:1]).numpy()

    probe = 32
    _run(probe)  # trace + warm up outside the measurement
    tf.config.experimental.reset_memory_stats("GPU:0")
    base = tf.config.experimental.get_memory_info("GPU:0")["current"]
    _run(probe)
    peak = tf.config.experimental.get_memory_info("GPU:0")["peak"]
    per_row = max(peak - base, 1) / probe
    rows = int(SAMPLER_VRAM_BUDGET_MIB * 2**20 // per_row)
    rows = max(8, min(rows, MAX_SAMPLE_ROWS))
    # Confirm the size with one real step (allocator fragmentation and cuDNN workspaces grow
    # faster than linearly); halve until it fits so the sampling loop itself never OOMs.
    while True:
        try:
            _run(rows)
            break
        except tf.errors.ResourceExhaustedError:
            if rows <= 8:
                raise
            rows //= 2
            gc.collect()
    _ROWS[name] = rows
    print(f"  [{name}] probe: {per_row / 2**20:.2f} MiB per row -> {rows} rows per GPU call")
    return rows


def _model_tag(model) -> str:
    """Checkpoint run directory of a model in TRAINED (ties a cache entry to its weights)."""
    for key, m in TRAINED.items():
        if m is model:
            run = next(r for r in TRAIN_META if (r["name"], r["seed"]) == key)
            return os.path.basename(run["run_dir"])
    raise KeyError("draw_ensemble: model is not one of TRAINED")


def _data_tag(img_hist, pv_hist) -> str:
    h = hashlib.sha1()
    h.update(np.ascontiguousarray(pv_hist).tobytes())
    h.update(np.ascontiguousarray(img_hist[:: max(1, len(img_hist) // 64)]).tobytes())
    h.update(str(img_hist.shape).encode())
    return h.hexdigest()[:10]


def draw_ensemble(
    name,
    model,
    img_hist,
    pv_hist,
    M: int,
    steps: int,
    batch_size: int = None,
    seed: int = 0,
):
    """Returns (ensemble_kW (N, Tf, M), seconds per member per window).

    batch_size: (window, member) rows per GPU call; None sizes it from free VRAM.
    Latency is GPU throughput: wall time of the whole ensemble / (N * M), with the members of
    many windows sampled together.
    """
    N, Tf = len(img_hist), model.cfg.fut_frames
    cache = os.path.join(
        ENS_CACHE_DIR,
        f"{_model_tag(model)}__{_data_tag(img_hist, pv_hist)}_N{N}_M{M}_S{steps}_r{seed}.npz",
    )
    if os.path.exists(cache):
        with np.load(cache) as z:
            return z["ens"], float(z["latency"])

    sampler = _sampler_for(name, model)
    coeffs = _ddim_coeffs(model, steps)
    rows = batch_size or _rows_for(name, sampler, img_hist, pv_hist, M, coeffs)

    out = np.empty((N * M, Tf), dtype=np.float32)
    t0 = time.time()
    s = 0
    while s < N * M:
        e = min(s + rows, N * M)
        flat = np.minimum(np.arange(s, s + rows), N * M - 1)  # padded: one traced shape
        try:
            pv_n = sampler.run_rows(img_hist, pv_hist, flat, M, seed, coeffs).numpy()
        except tf.errors.ResourceExhaustedError:
            if rows <= 8:
                raise
            rows = _ROWS[name] = rows // 2
            gc.collect()
            print(f"\n  [{name}] GPU OOM -> retrying with {rows} rows per call")
            continue
        out[s:e] = pv_n[: e - s, :, 0]
        s = e
        sys.stdout.write(f"\r  [{name}] {s // M}/{N} windows sampled")
        sys.stdout.flush()
    elapsed = time.time() - t0
    sys.stdout.write("\r" + " " * 60 + "\r")

    ens = np.clip(denormalize_pv(out), 0.0, PV_RATED_CAPACITY_KW)
    ens = ens.reshape(N, M, Tf).transpose(0, 2, 1).astype(np.float32)  # (N, Tf, M)
    latency = elapsed / max(N * M, 1)
    tmp = cache[:-4] + ".tmp.npz"
    np.savez(tmp, ens=ens, latency=latency)
    os.replace(tmp, cache)
    return ens, latency


def score_ensemble(y_kw, ens_kw, day_ids, levels=None, n_boot=None) -> Dict[str, Any]:
    """Full probabilistic scorecard for one ensemble."""
    levels = levels or Run.nominal_levels
    n_boot = n_boot or Run.n_boot
    per_case_fair = crps_per_case(y_kw, ens_kw, fair=True)
    per_case_bias = crps_per_case(y_kw, ens_kw, fair=False)
    i90 = interval_stats(y_kw, ens_kw, 0.9)
    flat = ens_kw.reshape(-1, ens_kw.shape[-1])
    return {
        "crps_fair": float(np.nanmean(per_case_fair)),
        "crps_biased": float(np.nanmean(per_case_bias)),
        "crps_per_case": per_case_fair,
        "crps_per_lead": np.nanmean(per_case_fair, axis=0).tolist(),
        "mae_of_median": float(np.nanmean(np.abs(np.median(ens_kw, axis=-1) - y_kw))),
        "winkler_90": i90["winkler"],
        "coverage_90": i90["coverage"],
        "width_90": i90["width"],
        "n_members": int(ens_kw.shape[-1]),
        "bootstrap": day_block_bootstrap(per_case_fair, day_ids, n_boot=n_boot),
        "rank_histogram": rank_histogram(y_kw, flat),
        "spread_skill": spread_skill_ratio(y_kw, flat),
        "hersbach": hersbach_decomposition(y_kw, flat),
        "reliability": reliability_curve(y_kw, ens_kw, levels),
    }


# ---- Smart persistence reference (deterministic: its CRPS equals its MAE)
_spm = SmartPersistenceBaseline()
_spm_pred = _spm.predict(
    TEST["pv_hist_kw"], TEST["times_hist"][:, -1], TEST["times_fut"]
)
SPM = {
    "crps": crps_ensemble(TEST["pv_true_kw"], _spm_pred[..., None], fair=False),
    "per_case": crps_per_case(TEST["pv_true_kw"], _spm_pred[..., None], fair=False),
}
print(f"Smart persistence CRPS (= MAE, deterministic): {SPM['crps']:.3f} kW")


EVAL: Dict[str, Any] = {}
ENSEMBLES: Dict[str, np.ndarray] = {}

for _name in VARIANT_SPEC:
    per_seed = []
    for _seed in Run.seed_list:
        ens, lat = draw_ensemble(
            _name,
            TRAINED[(_name, _seed)],
            TEST["img_hist"],
            TEST["pv_hist"],
            M=Run.M_eval,
            steps=Run.steps_eval,
            seed=_seed,
        )
        sc = score_ensemble(TEST["pv_true_kw"], ens, TEST["day_ids"])
        sc["latency_s_per_member"] = lat
        sc["seed"] = _seed
        per_seed.append(sc)
        if _seed == Run.seed_list[0]:
            ENSEMBLES[_name] = ens
        print(
            f"  {_name} seed{_seed}: fair CRPS {sc['crps_fair']:.3f} kW | "
            f"WS90 {sc['winkler_90']:.2f} | cov90 {sc['coverage_90']:.3f} | "
            f"{1000*lat:.0f} ms/member"
        )
        gc.collect()
    EVAL[_name] = {
        "per_seed": per_seed,
        "crps_fair_mean": float(np.mean([s["crps_fair"] for s in per_seed])),
        "crps_fair_std": float(np.std([s["crps_fair"] for s in per_seed])),
        # per-case score averaged over seeds = expected performance of the training procedure
        "crps_per_case_mean": np.mean([s["crps_per_case"] for s in per_seed], axis=0),
        "params": int(np.mean([m["params"] for m in TRAIN_META if m["name"] == _name])),
        "train_seconds": float(
            np.mean([m["train_seconds"] for m in TRAIN_META if m["name"] == _name])
        ),
        "latency_s_per_member": float(
            np.mean([s["latency_s_per_member"] for s in per_seed])
        ),
    }

print()
print("Sampling complete.")

nvidia-smi: 5247 MiB free of 8192 MiB -> sampler budget 3672 MiB (70% of free)
Smart persistence CRPS (= MAE, deterministic): 1.357 kW
  [A0_joint_gen] probe: 32.77 MiB per row -> 112 rows per GPU call
  A0_joint_gen seed0: fair CRPS 0.955 kW | WS90 9.64 | cov90 0.786 | 63 ms/member
  A0_joint_gen seed1: fair CRPS 1.341 kW | WS90 12.25 | cov90 0.624 | 59 ms/member
  A0_joint_gen seed2: fair CRPS 0.910 kW | WS90 8.64 | cov90 0.774 | 58 ms/member
  [A1_persist_frame] probe: 13.02 MiB per row -> 141 rows per GPU call
  A1_persist_frame seed0: fair CRPS 0.896 kW | WS90 8.28 | cov90 0.673 | 26 ms/member
  A1_persist_frame seed1: fair CRPS 0.909 kW | WS90 8.21 | cov90 0.643 | 25 ms/member
  A1_persist_frame seed2: fair CRPS 0.875 kW | WS90 7.35 | cov90 0.718 | 25 ms/member
  [A2_cond_pv] probe: 0.13 MiB per row -> 16384 rows per GPU call
  A2_cond_pv seed0: fair CRPS 0.916 kW | WS90 7.93 | cov90 0.709 | 0 ms/member
  A2_cond_pv seed1: fair CRPS 0.894 kW | WS90 7.26 | cov90 0.751 | 0 ms/membe

In [13]:
# ---- RQ1: paired, day-block significance tests down the ablation ladder
LADDER_TESTS = {}
for a, b in [
    ("A0_joint_gen", "A1_persist_frame"),
    ("A0_joint_gen", "A2_cond_pv"),
    ("A1_persist_frame", "A2_cond_pv"),
    ("A2_cond_pv", "A3_pv_only"),
    ("A0_joint_gen", "A3_pv_only"),
]:
    LADDER_TESTS[f"{a} vs {b}"] = paired_day_block_test(
        EVAL[a]["crps_per_case_mean"],
        EVAL[b]["crps_per_case_mean"],
        TEST["day_ids"],
        n_boot=Run.n_boot,
    )

ladder_rows = []
for k, v in LADDER_TESTS.items():
    ladder_rows.append(
        {
            "comparison": k,
            "delta CRPS (kW)": f"{v['delta']:+.4f}",
            "delta (%)": f"{v['delta_pct']:+.2f}%",
            "95% CI": f"[{v['ci_low']:+.4f}, {v['ci_high']:+.4f}]",
            "p": f"{v['p_value']:.3f}",
            "significant": v["significant"],
        }
    )
df_ladder = pd.DataFrame(ladder_rows)
print("Negative delta = the FIRST model is better. Paired day-block bootstrap.")
print()
print(df_ladder.to_string(index=False))

_a0a1 = LADDER_TESTS["A0_joint_gen vs A1_persist_frame"]
_a0a2 = LADDER_TESTS["A0_joint_gen vs A2_cond_pv"]
print()
print("--- RQ1 / H1 verdict ---")
print("H1 predicted: A0 beats A2 by <= 5% fair CRPS and/or not significantly.")
print(f"  A0 vs A2: {_a0a2['delta_pct']:+.2f}%  significant={_a0a2['significant']}")
print(
    f"  A0 vs A1: {_a0a1['delta_pct']:+.2f}%  significant={_a0a1['significant']}"
    "   <- isolates future-sky GENERATION at matched capacity"
)
_h1_falsified = (_a0a2["delta_pct"] < -5.0) and _a0a2["significant"]
print(
    "  H1 "
    + (
        "FALSIFIED - the generation branch does earn its cost"
        if _h1_falsified
        else "SUPPORTED - the generation branch does not earn its cost here"
    )
)

Negative delta = the FIRST model is better. Paired day-block bootstrap.

                      comparison delta CRPS (kW) delta (%)             95% CI     p  significant
A0_joint_gen vs A1_persist_frame         +0.1753   +19.62% [+0.1327, +0.2193] 0.000         True
      A0_joint_gen vs A2_cond_pv         +0.1600   +17.61% [+0.1195, +0.2015] 0.000         True
  A1_persist_frame vs A2_cond_pv         -0.0153    -1.69% [-0.0346, -0.0005] 0.044         True
        A2_cond_pv vs A3_pv_only         -0.0853    -8.59% [-0.1238, -0.0512] 0.000         True
      A0_joint_gen vs A3_pv_only         +0.0746    +7.51% [+0.0057, +0.1412] 0.032         True

--- RQ1 / H1 verdict ---
H1 predicted: A0 beats A2 by <= 5% fair CRPS and/or not significantly.
  A0 vs A2: +17.61%  significant=True
  A0 vs A1: +19.62%  significant=True   <- isolates future-sky GENERATION at matched capacity
  H1 SUPPORTED - the generation branch does not earn its cost here


### 6.1 Cost accounting

The ablation is only meaningful with its costs attached. A skill gain of a fraction of a percent bought
with several times the sampling latency is a negative result for the architecture, not a win.


In [14]:
cost_rows = []
for name, spec in VARIANT_SPEC.items():
    e = EVAL[name]
    cost_rows.append(
        {
            "Variant": spec["label"],
            "Params": f"{e['params']:,}",
            "Train (s)": f"{e['train_seconds']:.0f}",
            "ms/member": f"{1000*e['latency_s_per_member']:.1f}",
            f"s @ M={Run.M_eval}": f"{Run.M_eval*e['latency_s_per_member']:.2f}",
            "Fair CRPS (kW)": f"{e['crps_fair_mean']:.3f} +/- {e['crps_fair_std']:.3f}",
            "Skill vs SPM": f"{100*(1 - e['crps_fair_mean']/SPM['crps']):+.1f}%",
        }
    )
df_cost = pd.DataFrame(cost_rows)
print(df_cost.to_string(index=False))
print()
print(f"Smart persistence CRPS = {SPM['crps']:.3f} kW.")
print(
    f"Operational budget: the forecast must be delivered inside the "
    f"{CONFIG_15M.cadence_minutes}-minute update cadence = {60*CONFIG_15M.cadence_minutes} s "
    "for the whole ensemble."
)

                             Variant  Params Train (s) ms/member s @ M=50  Fair CRPS (kW) Skill vs SPM
     A0 joint-gen (baseline reimpl.) 925,412     91459      60.1     3.00 1.069 +/- 0.193       +21.3%
A1 persist-frame (no sky generation) 925,412     57087      25.5     1.27 0.893 +/- 0.014       +34.2%
     A2 cond-pv (cached sky encoder) 974,817     29516       0.2     0.01 0.909 +/- 0.010       +33.0%
             A3 pv-only (no imagery) 362,593     16183       0.2     0.01 0.994 +/- 0.021       +26.8%

Smart persistence CRPS = 1.357 kW.
Operational budget: the forecast must be delivered inside the 2-minute update cadence = 120 s for the whole ensemble.


---

## 7. Contribution C2 — the calibration audit (RQ2)

This is the section that does not exist anywhere in the sky-image PV diffusion literature. The question is
not "is CRPS low" but "when this model says 90%, does it happen 90% of the time".


In [15]:
cal_rows = []
for name, spec in VARIANT_SPEC.items():
    s = EVAL[name]["per_seed"][0]
    rh, ss, hb, rc = (
        s["rank_histogram"],
        s["spread_skill"],
        s["hersbach"],
        s["reliability"],
    )
    cal_rows.append(
        {
            "Variant": spec["label"],
            "Spread/skill": f"{ss['ratio_corrected']:.3f}",
            "Rank RI": f"{rh['reliability_index']:.3f}",
            "RI null p95": f"{rh['ri_null_p95']:.3f}",
            "Flat?": rh["calibrated"],
            "Outliers": f"{rh['outlier_fraction']:.3f}",
            "(expected)": f"{rh['expected_outlier_fraction']:.3f}",
            "ACE": f"{rc['ace']:.3f}",
            "Signed ACE": f"{rc['signed_ace']:+.3f}",
            "Cov@90": f"{s['coverage_90']:.3f}",
            "Reliability": f"{hb['reliability']:.3f}",
            "Resolution": f"{hb['resolution']:.3f}",
            "Reli/CRPS": f"{100*hb['reliability']/max(hb['crps_reconstructed'],1e-9):.1f}%",
        }
    )
df_cal = pd.DataFrame(cal_rows)
print(df_cal.to_string(index=False))

print()
print("--- RQ2 / H2 verdict ---")
print(
    "H2 predicted MISCALIBRATION, specifically UNDERdispersion (spread/skill < 1, U-shaped"
)
print("rank histogram, non-trivial Hersbach reliability share).")
for name, spec in VARIANT_SPEC.items():
    s = EVAL[name]["per_seed"][0]
    ss, rh, hb = s["spread_skill"], s["rank_histogram"], s["hersbach"]
    share = 100 * hb["reliability"] / max(hb["crps_reconstructed"], 1e-9)
    if rh["calibrated"]:
        verdict = "calibrated (H2 falsified for this variant)"
    elif ss["underdispersed"]:
        verdict = "UNDERDISPERSED / overconfident (H2 supported)"
    else:
        verdict = "OVERDISPERSED / underconfident (miscalibrated, but opposite to H2)"
    print(
        f"  {spec['label']:<42s} spread/skill={ss['ratio_corrected']:.3f}  "
        f"flat={rh['calibrated']}  reliability share={share:.1f}%"
    )
    print(f"    -> {verdict}")

print()
print(
    "A low CRPS with a large reliability share means the score is being carried by sharpness"
)
print("while the intervals are wrong -- exactly what CRPS alone cannot reveal.")

                             Variant Spread/skill Rank RI RI null p95  Flat? Outliers (expected)   ACE Signed ACE Cov@90 Reliability Resolution Reli/CRPS
     A0 joint-gen (baseline reimpl.)        0.629   0.276       0.049  False    0.097      0.039 0.054     -0.054  0.786       0.050      3.310      5.2%
A1 persist-frame (no sky generation)        0.744   0.431       0.049  False    0.156      0.039 0.168     -0.168  0.673       0.033      3.350      3.6%
     A2 cond-pv (cached sky encoder)        0.860   0.406       0.049  False    0.116      0.039 0.159     -0.159  0.709       0.018      3.313      1.9%
             A3 pv-only (no imagery)        0.636   0.432       0.050  False    0.136      0.039 0.161     -0.161  0.678       0.055      3.261      5.4%

--- RQ2 / H2 verdict ---
H2 predicted MISCALIBRATION, specifically UNDERdispersion (spread/skill < 1, U-shaped
rank histogram, non-trivial Hersbach reliability share).
  A0 joint-gen (baseline reimpl.)            spread/skill=0.6

---

## 8. Contribution C5 — post-hoc recalibration (RQ3)

Two corrections, both fitted on the **validation days only** and applied unchanged to the test days.
Neither requires retraining, so both are deployable on top of any existing model in this literature.

1. **Spread inflation, fitted two ways.** Replace each member $x_i$ by
   $\bar{x} + b + \gamma (x_i - \bar{x})$. The factor is fitted on validation under **two different
   objectives**, because they need not agree and the disagreement is itself the point:
   - $\gamma_{\mathrm{CRPS}}$ minimises validation fair CRPS — what you would pick if CRPS is all you
     look at, which is what this sub-literature currently does;
   - $\gamma_{\mathrm{cal}}$ minimises validation average coverage error — what you would pick to make
     the intervals mean what they say.

   $\gamma_{\mathrm{cal}} > 1$ is direct evidence the raw ensemble was too **narrow** (overconfident);
   $< 1$ that it was too **wide**. Reporting both exposes the reliability–sharpness trade-off that a
   single CRPS number hides.

2. **Split-conformal intervals (CQR).** Take the ensemble's empirical
   $[\,q_{\alpha/2}, q_{1-\alpha/2}\,]$ interval, compute the conformity score
   $E = \max(L - y,\; y - U)$ on validation, and widen every test interval by its
   $\lceil (n+1)(1-\alpha)\rceil / n$ empirical quantile. This gives finite-sample marginal coverage
   with no distributional assumption (Romano, Patterson & Candès, 2019).


In [16]:
# Calibration set: the held-out validation DAYS, never seen in training
VAL = {
    "pv_true_kw": trainval_pv_raw[val_idx[1]],
    "pv_hist_kw": trainval_pv_raw[val_idx[0]],
    "img_hist": normalize_images(trainval_images_mmap[val_idx[0]]).astype(np.float32),
}
VAL["pv_hist"] = normalize_pv(VAL["pv_hist_kw"])[..., None].astype(np.float32)
print(f"Calibration (validation) windows: {len(VAL['pv_true_kw'])}")


def apply_inflation(ens, gamma, bias=0.0):
    m = np.mean(ens, axis=-1, keepdims=True)
    return np.clip(m + bias + gamma * (ens - m), 0.0, PV_RATED_CAPACITY_KW)


def fit_inflation(y_val, ens_val, objective: str = "crps", gammas=None, biases=None):
    """objective="crps": minimise validation fair CRPS.
    objective="ace":  minimise validation average coverage error.
    Returns (gamma, bias, objective_value)."""
    gammas = gammas if gammas is not None else np.arange(0.50, 3.01, 0.05)
    biases = biases if biases is not None else np.array([0.0])

    def _obj(e):
        if objective == "crps":
            return crps_ensemble(y_val, e, fair=True)
        return reliability_curve(y_val, e, Run.nominal_levels)["ace"]

    best = (1.0, 0.0, _obj(ens_val))
    for gm in gammas:
        for bs in biases:
            v = _obj(apply_inflation(ens_val, gm, bs))
            if v < best[2]:
                best = (float(gm), float(bs), float(v))
    return best


def conformal_widening(y_val, ens_val, alpha=0.10):
    L = np.percentile(ens_val, 100 * alpha / 2, axis=-1)
    U = np.percentile(ens_val, 100 * (1 - alpha / 2), axis=-1)
    E = np.maximum(L - y_val, y_val - U).reshape(-1)
    n = len(E)
    q_level = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(E, q_level))


RECAL = {}
for name, spec in VARIANT_SPEC.items():
    model = TRAINED[(name, Run.seed_list[0])]
    ens_val, _ = draw_ensemble(
        name,
        model,
        VAL["img_hist"],
        VAL["pv_hist"],
        M=Run.M_eval,
        steps=Run.steps_eval,
        seed=1234,
    )
    g_crps, b_crps, v_crps = fit_inflation(VAL["pv_true_kw"], ens_val, objective="crps")
    g_cal, b_cal, v_ace = fit_inflation(VAL["pv_true_kw"], ens_val, objective="ace")
    q_conf = conformal_widening(VAL["pv_true_kw"], ens_val, alpha=0.10)

    ens_te = ENSEMBLES[name]
    nb_small = max(200, Run.n_boot // 5)
    before = score_ensemble(
        TEST["pv_true_kw"], ens_te, TEST["day_ids"], n_boot=nb_small
    )
    after_crps = score_ensemble(
        TEST["pv_true_kw"],
        apply_inflation(ens_te, g_crps, b_crps),
        TEST["day_ids"],
        n_boot=nb_small,
    )
    after_cal = score_ensemble(
        TEST["pv_true_kw"],
        apply_inflation(ens_te, g_cal, b_cal),
        TEST["day_ids"],
        n_boot=nb_small,
    )

    # Conformalised 90% interval on the raw ensemble
    alpha = 0.10
    i90 = interval_stats(TEST["pv_true_kw"], ens_te, 1 - alpha)
    Lc = np.clip(i90["lower"] - q_conf, 0.0, PV_RATED_CAPACITY_KW)
    Uc = np.clip(i90["upper"] + q_conf, 0.0, PV_RATED_CAPACITY_KW)
    y = TEST["pv_true_kw"]
    cov_conf = float(np.mean((y >= Lc) & (y <= Uc)))
    wink_conf = float(
        np.mean(
            (Uc - Lc)
            + np.where(y < Lc, 2 * (Lc - y) / alpha, 0.0)
            + np.where(y > Uc, 2 * (y - Uc) / alpha, 0.0)
        )
    )

    RECAL[name] = dict(
        gamma_crps=g_crps,
        bias_crps=b_crps,
        val_crps=v_crps,
        gamma_cal=g_cal,
        bias_cal=b_cal,
        val_ace=v_ace,
        q_conformal=q_conf,
        before=before,
        after_crps=after_crps,
        after_cal=after_cal,
        coverage_90_conformal=cov_conf,
        winkler_90_conformal=wink_conf,
    )
    print(
        f"  {name}: gamma_CRPS={g_crps:.2f}  gamma_cal={g_cal:.2f}  "
        f"conformal widening {q_conf:.2f} kW"
    )
    gc.collect()

recal_rows = []
for name, spec in VARIANT_SPEC.items():
    r = RECAL[name]
    b, ac, aa = r["before"], r["after_crps"], r["after_cal"]
    recal_rows.append(
        {
            "Variant": spec["label"],
            "g_CRPS": f"{r['gamma_crps']:.2f}",
            "g_cal": f"{r['gamma_cal']:.2f}",
            "ACE raw": f"{b['reliability']['ace']:.3f}",
            "ACE g_CRPS": f"{ac['reliability']['ace']:.3f}",
            "ACE g_cal": f"{aa['reliability']['ace']:.3f}",
            "CRPS raw": f"{b['crps_fair']:.3f}",
            "CRPS g_CRPS": f"{ac['crps_fair']:.3f}",
            "CRPS g_cal": f"{aa['crps_fair']:.3f}",
            "Cov90 raw": f"{b['coverage_90']:.3f}",
            "Cov90 g_cal": f"{aa['coverage_90']:.3f}",
            "Cov90 conf.": f"{r['coverage_90_conformal']:.3f}",
            "S/S g_cal": f"{aa['spread_skill']['ratio_corrected']:.3f}",
        }
    )
df_recal = pd.DataFrame(recal_rows)
print()
print(df_recal.to_string(index=False))

print()
print("--- RQ3 / H3 verdict ---")
print(
    "H3 predicted: the calibration-fitted correction cuts ACE by >= 50% at <= 2% fair-CRPS cost."
)
for name, spec in VARIANT_SPEC.items():
    r = RECAL[name]
    ace_b = r["before"]["reliability"]["ace"]
    ace_a = r["after_cal"]["reliability"]["ace"]
    c_b, c_a = r["before"]["crps_fair"], r["after_cal"]["crps_fair"]
    ok = (ace_a <= 0.5 * ace_b) and (c_a <= 1.02 * c_b)
    print(
        f"  {spec['label']:<42s} ACE {ace_b:.3f} -> {ace_a:.3f} "
        f"({100*(ace_a-ace_b)/max(ace_b,1e-9):+.0f}%), CRPS {c_b:.3f} -> {c_a:.3f} "
        f"({100*(c_a-c_b)/c_b:+.1f}%)  H3 {'supported' if ok else 'not supported'}"
    )

print()
print("Reading the two gammas:")
print(
    "  gamma_cal > 1 -> the raw ensemble was too NARROW (overconfident); < 1 -> too WIDE."
)
print(
    "  Where gamma_CRPS and gamma_cal disagree, CRPS and calibration pull in opposite"
)
print("  directions -- exactly what a CRPS-only evaluation cannot reveal.")

Calibration (validation) windows: 11096
  A0_joint_gen: gamma_CRPS=1.40  gamma_cal=1.10  conformal widening 0.23 kW
  A1_persist_frame: gamma_CRPS=1.40  gamma_cal=1.55  conformal widening 0.26 kW
  A2_cond_pv: gamma_CRPS=1.30  gamma_cal=1.80  conformal widening 0.24 kW
  A3_pv_only: gamma_CRPS=1.50  gamma_cal=1.65  conformal widening 0.36 kW

                             Variant g_CRPS g_cal ACE raw ACE g_CRPS ACE g_cal CRPS raw CRPS g_CRPS CRPS g_cal Cov90 raw Cov90 g_cal Cov90 conf. S/S g_cal
     A0 joint-gen (baseline reimpl.)   1.40  1.10   0.054      0.050     0.028    0.955       0.924      0.943     0.786       0.818       0.857     0.690
A1 persist-frame (no sky generation)   1.40  1.55   0.168      0.054     0.019    0.896       0.870      0.872     0.673       0.849       0.868     1.124
     A2 cond-pv (cached sky encoder)   1.30  1.80   0.159      0.075     0.031    0.916       0.909      0.949     0.709       0.916       0.904     1.419
             A3 pv-only (no imagery

---

## 9. Contribution C4 — the latency × quality Pareto frontier (RQ4)

The baseline ablates the _training_ trajectory (500 vs 1000 steps, linear vs cosine). This sweeps the
_inference_ budget, which is the quantity an operator actually chooses, and — crucially — rescores
**calibration as well as CRPS** at every budget. H4 says calibration breaks before CRPS does; if so,
a step budget picked by CRPS alone (as the field currently does) silently ships an overconfident model.


In [17]:
STEP_SWEEP = {}
M_sweep_small = max(4, Run.M_eval // 2)
for name, spec in VARIANT_SPEC.items():
    model = TRAINED[(name, Run.seed_list[0])]
    sweep_rows = []
    for steps in Run.steps_sweep:
        ens, lat = draw_ensemble(
            name,
            model,
            TEST["img_hist"],
            TEST["pv_hist"],
            M=M_sweep_small,
            steps=steps,
            seed=7,
        )
        flat = ens.reshape(-1, ens.shape[-1])
        sweep_rows.append(
            {
                "steps": steps,
                "crps_fair": crps_ensemble(TEST["pv_true_kw"], ens, fair=True),
                "ace": reliability_curve(TEST["pv_true_kw"], ens, Run.nominal_levels)[
                    "ace"
                ],
                "reliability_index": rank_histogram(TEST["pv_true_kw"], flat)[
                    "reliability_index"
                ],
                "spread_skill": spread_skill_ratio(TEST["pv_true_kw"], flat)[
                    "ratio_corrected"
                ],
                "latency_ms_per_member": 1000 * lat,
                "latency_s_full_ensemble": Run.M_eval * lat,
            }
        )
        gc.collect()
    STEP_SWEEP[name] = sweep_rows
    d = pd.DataFrame(sweep_rows)
    d["crps_vs_largest_%"] = (
        100 * (d["crps_fair"] - d["crps_fair"].iloc[-1]) / d["crps_fair"].iloc[-1]
    )
    print()
    print(f"--- {spec['label']} (M={M_sweep_small}) ---")
    print(d.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print()
print("--- RQ4 / H4 verdict ---")
print("Threshold: CRPS within 2% of the largest budget; ACE within 0.02 of it.")
for name, spec in VARIANT_SPEC.items():
    d = pd.DataFrame(STEP_SWEEP[name])
    ref_c, ref_a = d["crps_fair"].iloc[-1], d["ace"].iloc[-1]
    ok_c = d[(d["crps_fair"] - ref_c) / ref_c <= 0.02]["steps"]
    ok_a = d[np.abs(d["ace"] - ref_a) <= 0.02]["steps"]
    min_c = int(ok_c.min()) if len(ok_c) else None
    min_a = int(ok_a.min()) if len(ok_a) else None
    if min_c is None or min_a is None:
        note = "inconclusive at these budgets"
    elif min_a > min_c:
        note = "calibration breaks FIRST (H4 supported)"
    elif min_a == min_c:
        note = "both hold to the same budget (H4 not supported here)"
    else:
        note = "CRPS breaks first (H4 contradicted)"
    print(
        f"  {spec['label']:<42s} CRPS holds to {min_c} steps | calibration holds to {min_a} steps"
    )
    print(f"    -> {note}")
if len(Run.steps_sweep) < 3:
    print()
    print(
        "NOTE: fewer than 3 step budgets were swept, so this verdict is degenerate. "
        "Use the full Run.steps_sweep for the paper."
    )

                                                            
--- A0 joint-gen (baseline reimpl.) (M=25) ---
 steps  crps_fair    ace  reliability_index  spread_skill  latency_ms_per_member  latency_s_full_ensemble  crps_vs_largest_%
     1     7.6549 0.2283             0.9857        0.8000                 3.0884                   0.1544           725.0342
     2     2.4328 0.3241             1.0306        0.5484                 6.0470                   0.3024           162.2082
     5     1.3962 0.3654             1.1338        0.3840                14.9682                   0.7484            50.4823
    10     1.0210 0.0996             0.4112        0.5241                29.8472                   1.4924            10.0444
    20     0.9668 0.0718             0.2653        0.6294                59.4262                   2.9713             4.2023
    50     0.9316 0.0771             0.2363        0.7306               148.3725                   7.4186             0.4033
   100     0.9278

### 9.1 Ensemble size, and why cross-paper CRPS comparisons are not sound as reported

SkyGPT reports 10 scenarios; `ddim.ipynb` uses 30; other work uses other budgets. With the **biased**
estimator everyone uses, a larger ensemble is scored more leniently, so part of any reported improvement is
an artefact of the sampling budget. This sweep quantifies that artefact directly.


In [18]:
M_ROWS = []
for name, spec in VARIANT_SPEC.items():
    full = ENSEMBLES[name]
    rng = np.random.default_rng(0)
    for M in Run.M_sweep:
        if M > full.shape[-1]:
            continue
        sub = full[..., rng.permutation(full.shape[-1])[:M]]
        M_ROWS.append(
            {
                "Variant": spec["label"],
                "M": M,
                "fair CRPS": crps_ensemble(TEST["pv_true_kw"], sub, fair=True),
                "biased CRPS": crps_ensemble(TEST["pv_true_kw"], sub, fair=False),
            }
        )
df_M = pd.DataFrame(M_ROWS)
df_M["bias (%)"] = 100 * (df_M["biased CRPS"] - df_M["fair CRPS"]) / df_M["fair CRPS"]
print(df_M.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print("'bias (%)' is the penalty a small-ensemble method pays purely for being small.")
print(
    "Any cross-paper CRPS comparison at unequal M is confounded by at least this much."
)

                             Variant  M  fair CRPS  biased CRPS  bias (%)
     A0 joint-gen (baseline reimpl.)  2     0.9327       1.2150   30.2741
     A0 joint-gen (baseline reimpl.)  5     0.9443       1.0576   11.9956
     A0 joint-gen (baseline reimpl.) 10     0.9605       1.0178    5.9604
     A0 joint-gen (baseline reimpl.) 20     0.9559       0.9846    2.9952
     A0 joint-gen (baseline reimpl.) 30     0.9523       0.9714    2.0001
     A0 joint-gen (baseline reimpl.) 50     0.9549       0.9663    1.1976
A1 persist-frame (no sky generation)  2     0.8948       1.1970   33.7737
A1 persist-frame (no sky generation)  5     0.8979       1.0161   13.1632
A1 persist-frame (no sky generation) 10     0.8958       0.9553    6.6406
A1 persist-frame (no sky generation) 20     0.8921       0.9219    3.3392
A1 persist-frame (no sky generation) 30     0.8939       0.9137    2.2105
A1 persist-frame (no sky generation) 50     0.8962       0.9080    1.3263
     A2 cond-pv (cached sky encoder)  

### 9.2 Ramp stratum

Reported as a **stratum**, not as a contribution — 2026 already has a dedicated physics-guided ramp
forecaster (Wang & You, [arXiv:2605.03165](https://arxiv.org/abs/2605.03165)). The question here is narrower
and still open: _does the sky-generation branch pay off specifically when the sky is changing fast?_
That is the strongest remaining case for keeping it, so it deserves an explicit test.


In [19]:
ramp = TEST["is_ramp"]
ramp_rows = []
for name, spec in VARIANT_SPEC.items():
    ens = ENSEMBLES[name]
    pc = crps_per_case(TEST["pv_true_kw"], ens, fair=True)
    ramp_rows.append(
        {
            "Variant": spec["label"],
            "CRPS all": f"{np.nanmean(pc):.3f}",
            "CRPS ramp": f"{np.nanmean(pc[ramp]):.3f}" if ramp.any() else "n/a",
            "CRPS non-ramp": f"{np.nanmean(pc[~ramp]):.3f}" if (~ramp).any() else "n/a",
            "Cov@90 ramp": (
                f"{interval_stats(TEST['pv_true_kw'][ramp], ens[ramp], 0.9)['coverage']:.3f}"
                if ramp.sum() > 1
                else "n/a"
            ),
        }
    )
print(pd.DataFrame(ramp_rows).to_string(index=False))

if ramp.sum() > 5 and len(np.unique(TEST["day_ids"][ramp])) > 2:
    rt = paired_day_block_test(
        crps_per_case(TEST["pv_true_kw"], ENSEMBLES["A0_joint_gen"], fair=True)[ramp],
        crps_per_case(TEST["pv_true_kw"], ENSEMBLES["A2_cond_pv"], fair=True)[ramp],
        TEST["day_ids"][ramp],
        n_boot=max(200, Run.n_boot // 5),
    )
    print()
    print(
        f"On ramp windows only, A0 vs A2: {rt['delta_pct']:+.2f}% "
        f"(95% CI [{rt['ci_low']:+.3f}, {rt['ci_high']:+.3f}], p={rt['p_value']:.3f}, "
        f"significant={rt['significant']})"
    )
    print("If the generation branch is worth keeping anywhere, it is here.")
else:
    print()
    print(
        "Too few ramp windows/days in this run to test the ramp stratum; enlarge the test set."
    )

                             Variant CRPS all CRPS ramp CRPS non-ramp Cov@90 ramp
     A0 joint-gen (baseline reimpl.)    0.955     4.292         0.631       0.464
A1 persist-frame (no sky generation)    0.896     3.952         0.600       0.589
     A2 cond-pv (cached sky encoder)    0.916     3.989         0.618       0.683
             A3 pv-only (no imagery)    1.008     4.589         0.661       0.459

On ramp windows only, A0 vs A2: +7.60% (95% CI [+0.065, +0.546], p=0.010, significant=True)
If the generation branch is worth keeping anywhere, it is here.


---

## 10. Figures and paper-ready tables


In [20]:
import matplotlib.pyplot as plt

short = ["A0\njoint-gen", "A1\npersist", "A2\ncond-pv", "A3\npv-only"]
colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd"]
tags = [n.split("_")[0] for n in VARIANT_SPEC]

fig = plt.figure(figsize=(16, 13))

# (a) CRPS with day-block bootstrap CIs
ax = fig.add_subplot(3, 2, 1)
means = [EVAL[n]["crps_fair_mean"] for n in VARIANT_SPEC]
los = [EVAL[n]["per_seed"][0]["bootstrap"]["ci_low"] for n in VARIANT_SPEC]
his = [EVAL[n]["per_seed"][0]["bootstrap"]["ci_high"] for n in VARIANT_SPEC]
ax.bar(short, means, color=colors, edgecolor="black")
ax.errorbar(
    short,
    means,
    yerr=[
        np.maximum(np.array(means) - los, 0),
        np.maximum(np.array(his) - np.array(means), 0),
    ],
    fmt="none",
    ecolor="black",
    capsize=5,
    lw=1.5,
)
ax.axhline(
    SPM["crps"], ls="--", c="red", label=f"smart persistence ({SPM['crps']:.2f})"
)
ax.set_ylabel("fair CRPS (kW)")
ax.set_title(
    "(a) RQ1: skill down the ablation ladder\n(95% day-block bootstrap CI)",
    fontweight="bold",
)
ax.legend(fontsize=8)

# (b) rank histograms
ax = fig.add_subplot(3, 2, 2)
for n, c, t in zip(VARIANT_SPEC, colors, tags):
    f = EVAL[n]["per_seed"][0]["rank_histogram"]["frequencies"]
    ax.plot(np.linspace(0, 1, len(f)), f, "o-", color=c, ms=3, label=t)
ax.axhline(1.0 / (Run.M_eval + 1), ls="--", c="k", label="calibrated")
ax.set_xlabel("normalised verification rank")
ax.set_ylabel("frequency")
ax.set_title(
    "(b) RQ2: rank histogram\nU-shape = overconfident, hump = underconfident",
    fontweight="bold",
)
ax.legend(fontsize=8)

# (c) reliability diagram
ax = fig.add_subplot(3, 2, 3)
ax.plot([0, 1], [0, 1], "k--", label="perfect")
for n, c, t in zip(VARIANT_SPEC, colors, tags):
    rc = EVAL[n]["per_seed"][0]["reliability"]
    ax.plot(rc["levels"], rc["coverage"], "o-", color=c, label=t)
ax.set_xlabel("nominal coverage")
ax.set_ylabel("empirical coverage")
ax.set_title(
    "(c) RQ2: reliability diagram\nbelow the diagonal = overconfident",
    fontweight="bold",
)
ax.legend(fontsize=8)

# (d) Hersbach decomposition
ax = fig.add_subplot(3, 2, 4)
pot = [EVAL[n]["per_seed"][0]["hersbach"]["crps_potential"] for n in VARIANT_SPEC]
rel = [EVAL[n]["per_seed"][0]["hersbach"]["reliability"] for n in VARIANT_SPEC]
ax.bar(
    short, pot, label="potential CRPS (information)", color="#8ecae6", edgecolor="black"
)
ax.bar(
    short,
    rel,
    bottom=pot,
    label="reliability (miscalibration)",
    color="#e63946",
    edgecolor="black",
)
ax.set_ylabel("kW")
ax.set_title("(d) RQ2: how much of CRPS is miscalibration?", fontweight="bold")
ax.legend(fontsize=8)

# (e) latency x CRPS Pareto
ax = fig.add_subplot(3, 2, 5)
for n, c, t in zip(VARIANT_SPEC, colors, tags):
    d = pd.DataFrame(STEP_SWEEP[n])
    ax.plot(d["latency_s_full_ensemble"], d["crps_fair"], "o-", color=c, label=t)
    for _, r in d.iterrows():
        ax.annotate(
            f"{int(r['steps'])}",
            (r["latency_s_full_ensemble"], r["crps_fair"]),
            fontsize=7,
            xytext=(3, 3),
            textcoords="offset points",
        )
ax.axvline(
    60 * CONFIG_15M.cadence_minutes,
    ls=":",
    c="red",
    label=f"{CONFIG_15M.cadence_minutes}-min update budget",
)
ax.set_xscale("log")
ax.set_xlabel(f"seconds for a full M={Run.M_eval} ensemble (log)")
ax.set_ylabel("fair CRPS (kW)")
ax.set_title(
    "(e) RQ4: latency x skill Pareto\n(labels = DDIM steps)", fontweight="bold"
)
ax.legend(fontsize=8)

# (f) does calibration degrade before CRPS?
ax = fig.add_subplot(3, 2, 6)
n0 = "A0_joint_gen"
d = pd.DataFrame(STEP_SWEEP[n0])
ax.plot(
    d["steps"],
    d["crps_fair"] / d["crps_fair"].iloc[-1],
    "o-",
    c="#1f77b4",
    label="fair CRPS (rel.)",
)
ax2 = ax.twinx()
ax2.plot(d["steps"], d["ace"], "s--", c="#e63946", label="ACE (calibration error)")
ax.set_xscale("log")
ax.set_xlabel("DDIM steps (log)")
ax.set_ylabel("fair CRPS relative to the largest budget")
ax2.set_ylabel("average coverage error")
ax.set_title("(f) RQ4: A0 -- does calibration break before CRPS?", fontweight="bold")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="center right")

plt.tight_layout()
_figpath = os.path.join(Run.figure_dir, "fig_main.png")
plt.savefig(_figpath, dpi=160, bbox_inches="tight")
plt.close(fig)
print(f"Saved {_figpath}")

Saved figures_gap/fig_main.png


In [21]:
def to_latex(df: pd.DataFrame, caption: str, label: str) -> str:
    body = df.to_latex(
        index=False, escape=True, column_format="l" + "r" * (len(df.columns) - 1)
    )
    return (
        "\\begin{table}[t]\n\\centering\n\\small\n"
        f"\\caption{{{caption}}}\n\\label{{tab:{label}}}\n{body}\\end{{table}}\n"
    )


LATEX = {
    "ablation": to_latex(
        df_cost,
        "Ablation ladder: probabilistic skill reported with its cost. "
        "Fair (unbiased) CRPS, mean $\\pm$ s.d. over training seeds.",
        "ablation",
    ),
    "significance": to_latex(
        df_ladder,
        "Paired day-block bootstrap tests down the ablation ladder. "
        "Negative $\\Delta$ favours the first model.",
        "significance",
    ),
    "calibration": to_latex(
        df_cal,
        "Calibration audit. Spread/skill $<1$ with a non-flat rank histogram "
        "indicates overconfidence; $>1$ indicates underconfidence.",
        "calibration",
    ),
    "recalibration": to_latex(
        df_recal,
        "Post-hoc recalibration fitted on validation days only, under a "
        "CRPS objective and a calibration objective.",
        "recal",
    ),
    "ensemble_size": to_latex(
        df_M.round(4), "Fair vs. biased CRPS against ensemble size $M$.", "msweep"
    ),
}
with open("tables_gap.tex", "w", encoding="utf-8") as fh:
    fh.write("% Generated by ddim_gap.ipynb -- do not edit by hand\n")
    fh.write(f"% QUICK_MODE = {QUICK_MODE}\n\n")
    for k, v in LATEX.items():
        fh.write(f"% ---- {k} ----\n{v}\n")
print("Wrote tables_gap.tex")

_DROP = {
    "crps_per_case",
    "crps_per_case_mean",
    "per_case",
    "ens_inflated",
    "lower",
    "upper",
    "winkler_per_case",
}


def _jsonable(o):
    if isinstance(o, dict):
        return {k: _jsonable(v) for k, v in o.items() if k not in _DROP}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    return o


RESULTS = {
    "quick_mode": QUICK_MODE,
    "generated_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "config": {
        k: v for k, v in vars(Run).items() if not k.startswith("_") and not callable(v)
    },
    "n_test_windows": int(len(TEST["pv_true_kw"])),
    "n_test_days": int(len(np.unique(TEST["day_ids"]))),
    "ramp_fraction": float(TEST["is_ramp"].mean()),
    "smart_persistence_crps": SPM["crps"],
    "training": _jsonable(TRAIN_META),
    "evaluation": _jsonable(EVAL),
    "ladder_tests": _jsonable(LADDER_TESTS),
    "recalibration": _jsonable(RECAL),
    "step_sweep": _jsonable(STEP_SWEEP),
    "ensemble_size_sweep": _jsonable(df_M.to_dict(orient="records")),
}
with open(Run.results_path, "w", encoding="utf-8") as fh:
    json.dump(RESULTS, fh, indent=2, default=str)
print(f"Wrote {Run.results_path}")
if QUICK_MODE:
    print()
    print(
        "*** QUICK_MODE run -- not publishable numbers. Re-run with QUICK_MODE = False. ***"
    )

ImportError: `Import Jinja2` failed. DataFrame.style requires jinja2. Use pip or conda to install the Jinja2 package.

---

## 11. Limitations and threats to validity

Written before the results, and to be kept in the paper whatever the results say.

1. **One site, one horizon.** SKIPP'D is a single 30 kW rooftop array in one Mediterranean climate at one
   15-minute horizon. A conclusion about the sky-generation branch here need not transfer to a site with
   different cloud regimes, or to 30/60-minute horizons where cloud advection has more time to matter.
   The honest framing is _"on the benchmark this sub-literature actually reports on"_.
2. **Capacity control differs across the ladder.** A0 and A1 are matched exactly (asserted at run time),
   so A0 vs. A1 is a clean test of sky _generation_. A2 is sized to be parameter-comparable but is not
   matched to the digit, so A0 vs. A2 is a _where-the-compute-goes_ comparison rather than a strict
   capacity-controlled one; A3 is deliberately small and is a floor, not a competitor. Each comparison
   must be reported with the control that actually applies to it.
3. **Training budget, not convergence.** Variants are matched on epochs and early stopping, not trained to
   convergence with per-variant hyperparameter search. A larger model given a fixed epoch budget may be
   undertrained — which, if A0 loses, is a live alternative explanation the paper must state. The partial
   defence is the seed replication and the reported validation losses.
4. **Deterministic DDIM sampling ($\eta = 0$).** Ensemble members then differ only through the initial
   Gaussian draw, which is itself a plausible contributor to miscalibration. `ddim.ipynb` already flags
   this in `DDIMSampler`. An $\eta > 0$ sweep is the obvious follow-up and is **not** run here; the
   calibration claims are therefore claims about the standard deterministic-sampler configuration.
5. **Test-set size and dependence.** Windows overlap within a day. Day-block bootstrapping handles the
   dependence but cannot manufacture statistical power from few days; with a small test set, wide CIs are
   the honest outcome, and "not significant" must not be reported as "no difference".
6. **We do not reproduce the baseline's published numbers.** Different split, stride, sampler, ensemble
   size and CRPS estimator. The paper compares variants _to each other under one protocol_; it must not
   claim to have replicated or refuted PV-MM-diffusion's reported 2.63 kW.
7. **Conformal coverage is marginal, not conditional.** Split conformal guarantees coverage averaged over
   the calibration distribution, not within ramp windows specifically, and assumes exchangeability between
   validation and test days — which seasonal drift can violate.

## 12. Paper outline

| Section              | Content                                                                                                                | Source |
| -------------------- | ---------------------------------------------------------------------------------------------------------------------- | ------ |
| 1 Introduction       | Ultra-short-term PV needs calibrated intervals; generative models are the 2024–2026 answer; two things are unverified. | §1     |
| 2 Related work       | SkyGPT, PV-MM-diffusion, the 2025 benchmarking result, ramp forecasting; the disagreement in the table in §1.1.        | §1.1   |
| 3 Problem & protocol | SKIPP'D, day-wise splits, fair CRPS, Hersbach, rank histograms, day-block resampling.                                  | §4     |
| 4 Ablation design    | The A0–A3 ladder and why A0 vs A1 is the decisive comparison.                                                          | §5     |
| 5 Results            | RQ1 table + significance; RQ2 calibration audit; RQ4 Pareto; ramp stratum.                                             | §6–9   |
| 6 Recalibration      | Two gammas + split conformal; the recipe others can adopt without retraining.                                          | §8     |
| 7 Limitations        | Verbatim from §11.                                                                                                     | §11    |
| 8 Conclusion         | What the field should stop paying for, and what it should start measuring.                                             | —      |

**Figures:** Fig. 1 ablation ladder with CIs · Fig. 2 rank histograms + reliability diagram ·
Fig. 3 Hersbach decomposition · Fig. 4 latency × skill Pareto. All emitted by §10.

**Venue fit.** The result is an empirical/negative-result audit, so it fits venues that reward those:
NeurIPS **Climate Change AI** workshop, **IEEE PES General Meeting**, **IEEE PVSC**, **ICMLA**, EU PVSEC.
It is _not_ a fit for a venue that requires a new architecture and a new state of the art.

### Reproducibility checklist

- [ ] `QUICK_MODE = False`, all seeds run
- [ ] `results_gap.json`, `tables_gap.tex`, `figures_gap/fig_main.png` regenerated from that run
- [ ] scoring self-tests (§4.1) passing in the same session
- [ ] `ddim.ipynb` commit hash recorded alongside the results
- [ ] SKIPP'D version and the exact day-level split recorded
- [ ] every claim in the paper traceable to a printed cell above

### What this notebook does **not** claim

It does not claim a new state of the art, does not claim to have replicated PV-MM-diffusion's published
numbers, and does not claim generality beyond SKIPP'D at a 15-minute horizon. Its contribution is a
question the field left open, answered under a protocol stricter than the one the field currently uses.
